In [1]:
import cv2
import numpy as np

# ============================================================
# 2x GoPro Stereo Calibration (FAST + ROBUST)
# - 4K videolarda bellek/süre sorununu azaltmak için: downscale + stride
# - Timestamp yerine: chessboard motion ile lag (offset) tahmini
# - Lag sweep ile en iyi lag seçimi
# - Epipolar distance ile outlier pair temizleme
# - Final stereo calibration + npz kaydı
# ============================================================

# ================== INPUTS ==================
LEFT_VIDEO  = "calib7_left.mp4"
RIGHT_VIDEO = "calib7_right.mp4"

# Chessboard: iç köşe sayısı (cols, rows)
PATTERN = (6, 5)
SQUARE_SIZE_MM = 130.0

# ================== SPEED SETTINGS ==================
SCALE = 0.5          # 4K için chessboard detection downscale
STRIDE_DET = 4       # her 4 frame'de bir dene (hız için)
MIN_SEP_DET = 4      # detection listesinde birbirine çok yakın frameleri seyrekleştir

# Lag sweep / Final
LAG_SWEEP_HALF = 20  # lag0 etrafında +/- 20 tarama (daha güvenli)
MAX_PAIRS_SWEEP = 15 # sweep hızlı olsun (15-25 arası iyi)
MAX_PAIRS_FINAL = 60 # finalde kullanılacak max pair (40-80 arası deneyebilirsin)

# Optimizer kriterleri (hız için iter azaltıldı)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 1e-6)

# Outlier temizleme agresifliği
OUTLIER_DROP_Q = 0.70  # en kötü %30'u at (0.70 -> keep best 70%)

# ================== OBJECT POINTS ==================
objp = np.zeros((PATTERN[0] * PATTERN[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:PATTERN[0], 0:PATTERN[1]].T.reshape(-1, 2)
objp *= SQUARE_SIZE_MM


# ================== HELPERS ==================
def find_corners_scaled(gray, scale=SCALE):
    """
    Chessboard corners'ı downscale görüntüde bulur,
    sonra koordinatları orijinal çözünürlüğe geri ölçekler.
    """
    if scale != 1.0:
        small = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    else:
        small = gray

    ok, corners = cv2.findChessboardCornersSB(
        small, PATTERN, flags=cv2.CALIB_CB_NORMALIZE_IMAGE
    )
    if not ok:
        return None

    corners = corners.astype(np.float32)
    if scale != 1.0:
        corners /= scale
    return corners


def collect_detections(video_path, stride=STRIDE_DET):
    """
    Video içinde chessboard görülen frameleri toplar.
    det item: (frame_idx, corners, img_size, centroid)
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Video açılamadı: {video_path}")

    det = []
    k = 0
    i = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if (k % stride) != 0:
            k += 1
            i += 1
            continue
        k += 1

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        corners = find_corners_scaled(gray)
        if corners is not None:
            cen = corners.reshape(-1, 2).mean(axis=0)
            det.append((i, corners, gray.shape[::-1], cen))

        i += 1

    cap.release()

    # yakındaki detectionları seyrekleştir (çeşitlilik)
    filtered = []
    last_i = -10**9
    for item in det:
        if item[0] - last_i >= MIN_SEP_DET:
            filtered.append(item)
            last_i = item[0]

    return filtered


def estimate_lag(detL, detR):
    """
    centroid hareket imzası üzerinden cross-correlation ile lag tahmini.
    lag: det-list index bazında (R, L'e göre kaç adım kayık?)
    """
    cenL = np.array([d[3] for d in detL], dtype=np.float64)
    cenR = np.array([d[3] for d in detR], dtype=np.float64)

    if len(cenL) < 5 or len(cenR) < 5:
        raise RuntimeError("Lag tahmini için yeterli detection yok (en az ~5).")

    sigL = np.linalg.norm(np.diff(cenL, axis=0), axis=1)
    sigR = np.linalg.norm(np.diff(cenR, axis=0), axis=1)

    sigL = (sigL - sigL.mean()) / (sigL.std() + 1e-9)
    sigR = (sigR - sigR.mean()) / (sigR.std() + 1e-9)

    corr = np.correlate(sigR, sigL, mode="full")
    lag = int(np.argmax(corr) - (len(sigL) - 1))
    return lag


def match_by_lag(detL, detR, lag, max_pairs):
    """
    det list index'e göre eşleştirir:
    iR = iL + lag
    """
    pairs = []
    img_size = detL[0][2]

    for iL in range(len(detL)):
        iR = iL + lag
        if 0 <= iR < len(detR):
            pairs.append((detL[iL][1], detR[iR][1], img_size))
            if len(pairs) >= max_pairs:
                break
    return pairs


def build_points(pairs):
    """pairs -> objpoints/imgpointsL/imgpointsR"""
    objpoints, imgpointsL, imgpointsR = [], [], []
    for cL, cR, _ in pairs:
        objpoints.append(objp.copy())
        imgpointsL.append(cL)
        imgpointsR.append(cR)
    return objpoints, imgpointsL, imgpointsR


def stereo_rms_only(pairs):
    """
    Lag sweep aşamasında hızlıca RMS kıyaslamak için.
    Burada FIX_INTRINSIC kullanıyoruz (daha stabil ve hızlı).
    """
    objpoints, imgpointsL, imgpointsR = build_points(pairs)
    img_size = pairs[0][2]

    _, K1, d1, *_ = cv2.calibrateCamera(objpoints, imgpointsL, img_size, None, None, criteria=criteria)
    _, K2, d2, *_ = cv2.calibrateCamera(objpoints, imgpointsR, img_size, None, None, criteria=criteria)

    retS, *_ = cv2.stereoCalibrate(
        objpoints, imgpointsL, imgpointsR,
        K1, d1, K2, d2,
        img_size, criteria=criteria,
        flags=cv2.CALIB_FIX_INTRINSIC
    )
    return retS


def stereo_final(pairs):
    """
    Final stereo calibration:
    - intrinsics calibrate
    - stereoCalibrate with USE_INTRINSIC_GUESS
    """
    objpoints, imgpointsL, imgpointsR = build_points(pairs)
    img_size = pairs[0][2]

    retL, K1, d1, *_ = cv2.calibrateCamera(objpoints, imgpointsL, img_size, None, None, criteria=criteria)
    retR, K2, d2, *_ = cv2.calibrateCamera(objpoints, imgpointsR, img_size, None, None, criteria=criteria)

    retS, K1, d1, K2, d2, R, T, E, F = cv2.stereoCalibrate(
        objpoints, imgpointsL, imgpointsR,
        K1, d1, K2, d2,
        img_size, criteria=criteria,
        flags=cv2.CALIB_USE_INTRINSIC_GUESS
    )
    return retL, retR, retS, K1, d1, K2, d2, R, T, E, F, img_size


def epipolar_errors(pairs, F):
    """
    Her pair için ortalama epipolar distance (pixel).
    (Left pts -> Right epipolar line) uzaklığı
    """
    errs = []
    for cL, cR, _ in pairs:
        ptsL = cL.reshape(-1, 2)
        ptsR = cR.reshape(-1, 2)

        ones = np.ones((ptsL.shape[0], 1), dtype=np.float64)
        x = np.hstack([ptsL.astype(np.float64), ones])  # Nx3

        l = (F @ x.T).T  # Nx3 (a,b,c)
        a, b, c = l[:, 0], l[:, 1], l[:, 2]
        x2, y2 = ptsR[:, 0].astype(np.float64), ptsR[:, 1].astype(np.float64)

        dist = np.abs(a * x2 + b * y2 + c) / (np.sqrt(a * a + b * b) + 1e-12)
        errs.append(float(dist.mean()))
    return np.array(errs, dtype=np.float64)


# ================== RUN ==================
detL = collect_detections(LEFT_VIDEO)
detR = collect_detections(RIGHT_VIDEO)

print(f"Detections: Left={len(detL)}, Right={len(detR)}")
if len(detL) < 20 or len(detR) < 20:
    print("Uyarı: Detections düşük. STRIDE_DET=2 veya 1 yapıp tekrar deneyebilirsin.")

lag0 = estimate_lag(detL, detR)
print("Initial lag estimate:", lag0)

# ---- LAG SWEEP (FAST) ----
best_rms = 1e9
best_lag = None

for lag_try in range(lag0 - LAG_SWEEP_HALF, lag0 + LAG_SWEEP_HALF + 1):
    pairs_try = match_by_lag(detL, detR, lag_try, MAX_PAIRS_SWEEP)
    if len(pairs_try) < 15:
        continue
    rms = stereo_rms_only(pairs_try)
    if rms < best_rms:
        best_rms = rms
        best_lag = lag_try

if best_lag is None:
    raise RuntimeError("Lag sweep başarısız. Detection az olabilir. STRIDE_DET düşürmeyi dene.")

print(f"Best lag (sweep): {best_lag} | Stereo RMS (sweep, ~{MAX_PAIRS_SWEEP} pairs): {best_rms}")

# ---- FINAL: more pairs with best lag ----
pairs_final = match_by_lag(detL, detR, best_lag, MAX_PAIRS_FINAL)
print("Pairs for final:", len(pairs_final))

retL, retR, retS, K1, d1, K2, d2, R, T, E, F, img_size = stereo_final(pairs_final)

print("=== PRE-CLEAN RESULTS ===")
print("Left RMS :", retL)
print("Right RMS:", retR)
print("Stereo RMS:", retS)
print("T (mm):", T.ravel())

# ---- EPIPOLAR OUTLIER CLEAN ----
errs = epipolar_errors(pairs_final, F)
thr = np.quantile(errs, OUTLIER_DROP_Q)  # keep best OUTLIER_DROP_Q fraction
pairs_clean = [p for p, e in zip(pairs_final, errs) if e <= thr]

print(f"Epipolar cleaning: keeping {len(pairs_clean)}/{len(pairs_final)} pairs (threshold={thr:.3f} px)")

# ---- FINAL CLEAN ----
retL2, retR2, retS2, K1c, d1c, K2c, d2c, Rc, Tc, Ec, Fc, _ = stereo_final(pairs_clean)

print("=== FINAL CLEAN RESULTS ===")
print("Left RMS :", retL2)
print("Right RMS:", retR2)
print("Stereo RMS:", retS2)
print("T (mm):", Tc.ravel())

# ---- SAVE ----
np.savez(
    "stereo_calib_final.npz",
    K1=K1c, d1=d1c, K2=K2c, d2=d2c,
    R=Rc, T=Tc, E=Ec, F=Fc,
    img_size=np.array(img_size),
    best_lag=np.array([best_lag], dtype=np.int32),
    sweep_rms=np.array([best_rms], dtype=np.float64),
    epipolar_errs=errs
)

print("Saved: stereo_calib_final.npz")


Detections: Left=24, Right=246
Initial lag estimate: 180
Best lag (sweep): 200 | Stereo RMS (sweep, ~15 pairs): 12.381283921970788
Pairs for final: 24
=== PRE-CLEAN RESULTS ===
Left RMS : 0.4275905806772612
Right RMS: 0.38185010195814983
Stereo RMS: 8.57839340127836
T (mm): [ 343.52716505 2008.7235104   222.42156609]
Epipolar cleaning: keeping 17/24 pairs (threshold=77.956 px)
=== FINAL CLEAN RESULTS ===
Left RMS : 0.44191867764851195
Right RMS: 0.39033465042547627
Stereo RMS: 7.603455867315451
T (mm): [ 192.855166   2571.44054683 -198.04173003]
Saved: stereo_calib_final.npz


In [2]:
import cv2
import numpy as np

# ============================================================
# 2x GoPro Stereo Calibration (ROBUST + WIDE-AWARE)
# - Preprocess (median + CLAHE) to suppress noise
# - Corner detection: SB -> fallback classic -> subpix refine
# - Strict validation: homography RMSE + reject near border
# - Lag estimation via centroid motion signature (approx frame-lag)
# - Matching by FRAME INDEX (nearest frame) with tolerance
# - Pair filtering via per-pair FundamentalMat RANSAC inlier ratio
# - Intrinsics: RATIONAL_MODEL (good for wide lenses)
# - Save npz
# ============================================================

# ================== INPUTS ==================
LEFT_VIDEO  = "calib7_left.mp4"
RIGHT_VIDEO = "calib7_right.mp4"

PATTERN = (6, 5)          # (cols, rows) inner corners
SQUARE_SIZE_MM = 130.0

# ================== SPEED / DETECTION ==================
SCALE = 0.5               # downscale for detection (4K)
STRIDE_CANDIDATES = [4, 2, 1]  # adaptive: if detections low, try smaller stride
MIN_DET_TARGET = 80       # aim for >=80 detections each side if possible
MIN_DET_ACCEPT = 20       # minimum workable
MIN_SEP_DET = 4           # avoid extremely close frames in detections (diversity)
MAX_DET = 600             # safety cap

# ================== LAG / PAIRS ==================
LAG_SWEEP_HALF = 30       # frames
MAX_PAIRS_SWEEP = 20
MAX_PAIRS_FINAL = 120
MATCH_TOL_FRAMES = 2      # nearest-frame tolerance when pairing

# ================== VALIDATION (FALSE POSITIVE SHIELD) ==================
HOMOGRAPHY_RMSE_MAX = 2.0  # px (strict)
BORDER_REJECT_PX = 30      # reject if corners too close to frame border

# ================== OPTIMIZATION ==================
SUBPIX_CRIT = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 1e-6)
CALIB_CRIT  = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 60, 1e-7)

# Wide lens support: add more distortion params
INTRINSIC_FLAGS = cv2.CALIB_RATIONAL_MODEL
# If needed, uncomment:
# INTRINSIC_FLAGS |= cv2.CALIB_THIN_PRISM_MODEL

# Stereo flags
STEREO_FLAGS_SWEEP = cv2.CALIB_FIX_INTRINSIC
STEREO_FLAGS_FINAL = cv2.CALIB_USE_INTRINSIC_GUESS

# RANSAC pair filtering
RANSAC_REPROJ_THR = 1.5
RANSAC_CONF = 0.999
PAIR_KEEP_FRAC = 0.80
PAIR_MIN_INLIER_FRAC = 0.65

# ================== OBJECT POINTS ==================
objp = np.zeros((PATTERN[0] * PATTERN[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:PATTERN[0], 0:PATTERN[1]].T.reshape(-1, 2)
objp *= SQUARE_SIZE_MM


# ================== HELPERS ==================
def preprocess_for_cb(gray):
    """Noise suppress + contrast boost (works well on real-world GoPro footage)."""
    g = cv2.medianBlur(gray, 3)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    g = clahe.apply(g)
    return g


def _find_corners(gray_small):
    """SB first, then classic fallback."""
    ok, corners = cv2.findChessboardCornersSB(
        gray_small, PATTERN, flags=cv2.CALIB_CB_NORMALIZE_IMAGE
    )
    if ok and corners is not None:
        return corners.astype(np.float32)

    ok, corners = cv2.findChessboardCorners(
        gray_small, PATTERN,
        flags=cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE
    )
    if ok and corners is not None:
        return corners.astype(np.float32)

    return None


def validate_corners_homography(corners, pattern, max_rmse_px=2.0):
    """
    Reject false positives by checking how well corners fit an ideal grid under homography.
    """
    cols, rows = pattern
    N = cols * rows
    if corners is None or len(corners) != N:
        return False, np.inf

    pts = corners.reshape(-1, 2).astype(np.float32)
    grid = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2).astype(np.float32)

    H, _ = cv2.findHomography(grid, pts, method=0)
    if H is None:
        return False, np.inf

    proj = cv2.perspectiveTransform(grid.reshape(-1, 1, 2), H).reshape(-1, 2)
    rmse = float(np.sqrt(np.mean(np.sum((proj - pts) ** 2, axis=1))))
    return (rmse <= max_rmse_px), rmse


def reject_if_near_border(corners, w, h, border=BORDER_REJECT_PX):
    pts = corners.reshape(-1, 2)
    if (pts[:, 0].min() < border or pts[:, 0].max() > (w - border) or
        pts[:, 1].min() < border or pts[:, 1].max() > (h - border)):
        return True
    return False


def find_corners_scaled(gray, scale=SCALE):
    """
    Detect corners on downscaled preprocessed image, map back to original,
    then subpixel refine and validate.
    """
    g = preprocess_for_cb(gray)

    if scale != 1.0:
        small = cv2.resize(g, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    else:
        small = g

    corners = _find_corners(small)
    if corners is None:
        return None

    if scale != 1.0:
        corners = corners / scale

    # subpixel refine on preprocessed original
    corners_ref = corners.copy()
    cv2.cornerSubPix(g, corners_ref, (11, 11), (-1, -1), SUBPIX_CRIT)

    h, w = gray.shape[:2]
    if reject_if_near_border(corners_ref, w, h):
        return None

    ok, rmse = validate_corners_homography(corners_ref, PATTERN, HOMOGRAPHY_RMSE_MAX)
    if not ok:
        return None

    return corners_ref


def collect_detections(video_path, stride):
    """
    Return list of dicts: {"frame": idx, "corners": ..., "size": (w,h), "centroid": ...}
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Video açılamadı: {video_path}")

    det = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if (frame_idx % stride) != 0:
            frame_idx += 1
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        corners = find_corners_scaled(gray)
        if corners is not None:
            cen = corners.reshape(-1, 2).mean(axis=0)
            h, w = gray.shape[:2]
            det.append({"frame": frame_idx, "corners": corners, "size": (w, h), "centroid": cen})
            if len(det) >= MAX_DET:
                break

        frame_idx += 1

    cap.release()

    # thin out detections to increase viewpoint diversity
    filtered = []
    last_f = -10**9
    min_sep_frames = MIN_SEP_DET * stride
    for d in det:
        if d["frame"] - last_f >= min_sep_frames:
            filtered.append(d)
            last_f = d["frame"]

    return filtered


def collect_with_adaptive_stride(video_path, label):
    best = None
    best_stride = None
    for stride in STRIDE_CANDIDATES:
        det = collect_detections(video_path, stride=stride)
        print(f"{label} detections (stride={stride}): {len(det)}")
        if best is None or len(det) > len(best):
            best, best_stride = det, stride
        if len(det) >= MIN_DET_TARGET:
            return det, best_stride
    return best, best_stride


def estimate_lag_frames(detL, detR):
    """
    Estimate lag in frames (R relative to L) using centroid motion signature.
    Converts detection-lag to frame-lag using median frame step.
    """
    if len(detL) < 6 or len(detR) < 6:
        raise RuntimeError("Lag tahmini için yeterli detection yok (>=6).")

    cenL = np.array([d["centroid"] for d in detL], dtype=np.float64)
    cenR = np.array([d["centroid"] for d in detR], dtype=np.float64)

    sigL = np.linalg.norm(np.diff(cenL, axis=0), axis=1)
    sigR = np.linalg.norm(np.diff(cenR, axis=0), axis=1)

    sigL = (sigL - sigL.mean()) / (sigL.std() + 1e-9)
    sigR = (sigR - sigR.mean()) / (sigR.std() + 1e-9)

    corr = np.correlate(sigR, sigL, mode="full")
    lag_det = int(np.argmax(corr) - (len(sigL) - 1))

    stepL = np.median(np.diff([d["frame"] for d in detL])) if len(detL) > 1 else 1
    stepR = np.median(np.diff([d["frame"] for d in detR])) if len(detR) > 1 else 1
    step = float(np.median([stepL, stepR]))

    return int(round(lag_det * step))


def match_by_frame_lag(detL, detR, lag_frames, max_pairs, tol_frames=MATCH_TOL_FRAMES):
    """
    Pair by actual frame index:
        frameR ~= frameL + lag_frames
    Nearest match within tol_frames.
    """
    mapR = {d["frame"]: d for d in detR}
    framesR = np.array(sorted(mapR.keys()), dtype=np.int64)

    pairs = []
    for dL in detL:
        target = dL["frame"] + lag_frames
        j = np.searchsorted(framesR, target)

        candidates = []
        if j < len(framesR):
            candidates.append(framesR[j])
        if j - 1 >= 0:
            candidates.append(framesR[j - 1])
        if not candidates:
            continue

        best_fr = min(candidates, key=lambda fr: abs(int(fr) - int(target)))
        if abs(int(best_fr) - int(target)) > tol_frames:
            continue

        dR = mapR[int(best_fr)]
        if dL["size"] != dR["size"]:
            continue

        pairs.append((dL["corners"], dR["corners"], dL["size"]))
        if len(pairs) >= max_pairs:
            break

    return pairs


def build_points(pairs):
    objpoints, imgpointsL, imgpointsR = [], [], []
    for cL, cR, _ in pairs:
        objpoints.append(objp.copy())
        imgpointsL.append(cL)
        imgpointsR.append(cR)
    return objpoints, imgpointsL, imgpointsR


def stereo_rms_only(pairs):
    if len(pairs) < 12:
        return np.inf
    objpoints, imgpointsL, imgpointsR = build_points(pairs)
    img_size = pairs[0][2]

    _, K1, d1, *_ = cv2.calibrateCamera(
        objpoints, imgpointsL, img_size, None, None,
        flags=INTRINSIC_FLAGS, criteria=CALIB_CRIT
    )
    _, K2, d2, *_ = cv2.calibrateCamera(
        objpoints, imgpointsR, img_size, None, None,
        flags=INTRINSIC_FLAGS, criteria=CALIB_CRIT
    )

    retS, *_ = cv2.stereoCalibrate(
        objpoints, imgpointsL, imgpointsR,
        K1, d1, K2, d2,
        img_size, criteria=CALIB_CRIT,
        flags=STEREO_FLAGS_SWEEP
    )
    return float(retS)


def stereo_final(pairs):
    if len(pairs) < 15:
        raise RuntimeError(f"Final stereo için pair az: {len(pairs)} (>=15 önerilir).")

    objpoints, imgpointsL, imgpointsR = build_points(pairs)
    img_size = pairs[0][2]

    retL, K1, d1, *_ = cv2.calibrateCamera(
        objpoints, imgpointsL, img_size, None, None,
        flags=INTRINSIC_FLAGS, criteria=CALIB_CRIT
    )
    retR, K2, d2, *_ = cv2.calibrateCamera(
        objpoints, imgpointsR, img_size, None, None,
        flags=INTRINSIC_FLAGS, criteria=CALIB_CRIT
    )

    retS, K1, d1, K2, d2, R, T, E, F = cv2.stereoCalibrate(
        objpoints, imgpointsL, imgpointsR,
        K1, d1, K2, d2,
        img_size, criteria=CALIB_CRIT,
        flags=STEREO_FLAGS_FINAL
    )
    return float(retL), float(retR), float(retS), K1, d1, K2, d2, R, T, E, F, img_size


def pair_inlier_score_F(cL, cR):
    ptsL = cL.reshape(-1, 2)
    ptsR = cR.reshape(-1, 2)
    F, mask = cv2.findFundamentalMat(
        ptsL, ptsR,
        method=cv2.FM_RANSAC,
        ransacReprojThreshold=RANSAC_REPROJ_THR,
        confidence=RANSAC_CONF,
        maxIters=5000
    )
    if F is None or mask is None:
        return 0.0
    return float(mask.sum()) / float(len(mask))


def ransac_filter_pairs(pairs, keep_frac=PAIR_KEEP_FRAC, min_inlier_frac=PAIR_MIN_INLIER_FRAC):
    scores = []
    for idx, (cL, cR, _) in enumerate(pairs):
        frac = pair_inlier_score_F(cL, cR)
        scores.append((frac, idx))

    scores.sort(reverse=True, key=lambda x: x[0])
    k = max(15, int(round(len(scores) * keep_frac)))

    kept = []
    kept_scores = []
    for frac, idx in scores[:k]:
        if frac >= min_inlier_frac:
            kept.append(pairs[idx])
            kept_scores.append(frac)

    # If too aggressive, relax threshold
    if len(kept) < 15 and len(pairs) >= 15:
        kept = [pairs[idx] for frac, idx in scores[:max(15, k)]]
        kept_scores = [frac for frac, idx in scores[:max(15, k)]]

    return kept, np.array(kept_scores, dtype=np.float64), np.array([s[0] for s in scores], dtype=np.float64)


def report_quality(retL, retR, retS, T, pair_scores):
    baseline = float(np.linalg.norm(T.reshape(-1)))
    med_inlier = float(np.median(pair_scores)) if len(pair_scores) else 0.0

    print("=== QUALITY REPORT ===")
    print(f"RMS L/R/S: {retL:.3f} / {retR:.3f} / {retS:.3f}")
    print(f"|T| (same unit as SQUARE_SIZE): {baseline:.3f}")
    print(f"Median pair inlier-frac: {med_inlier:.3f}")
    if retS > 3.0:
        print("[WARN] Stereo RMS yüksek -> eşleşme/false positive/lag sorunlu olabilir.")
    if baseline < 10.0:
        print("[WARN] Baseline çok küçük -> pairing/lag hatası olabilir.")
    if baseline > 1000.0:
        print("[WARN] Baseline aşırı büyük -> yanlış eşleşme/yanlış board parametreleri olabilir.")
    if med_inlier < 0.6:
        print("[WARN] RANSAC inlier düşük -> gürültü/yanlış pozitif/pair hatası olabilir.")


# ================== RUN ==================
detL, strideL = collect_with_adaptive_stride(LEFT_VIDEO, "LEFT")
detR, strideR = collect_with_adaptive_stride(RIGHT_VIDEO, "RIGHT")

print(f"Final detections: Left={len(detL)} (stride={strideL}), Right={len(detR)} (stride={strideR})")

if len(detL) < MIN_DET_ACCEPT or len(detR) < MIN_DET_ACCEPT:
    raise RuntimeError(
        "Detections çok az. Büyük ihtimalle sol video chessboard'u az görüyor "
        "veya validation çok sert. (BORDER_REJECT_PX azalt, HOMOGRAPHY_RMSE_MAX=2.5 dene, STRIDE=1)."
    )

lag0 = estimate_lag_frames(detL, detR)
print("Initial lag estimate (frames):", lag0)

# ---- LAG SWEEP ----
best_rms = np.inf
best_lag = None

for lag_try in range(lag0 - LAG_SWEEP_HALF, lag0 + LAG_SWEEP_HALF + 1):
    pairs_try = match_by_frame_lag(detL, detR, lag_try, MAX_PAIRS_SWEEP, tol_frames=MATCH_TOL_FRAMES)
    if len(pairs_try) < 12:
        continue
    rms = stereo_rms_only(pairs_try)
    if rms < best_rms:
        best_rms = rms
        best_lag = lag_try

if best_lag is None:
    raise RuntimeError("Lag sweep başarısız: yeterli pair oluşmuyor. MATCH_TOL_FRAMES=3 dene veya stride düşür.")

print(f"Best lag: {best_lag} frames | Sweep RMS (~{MAX_PAIRS_SWEEP} pairs): {best_rms:.4f}")

# ---- FINAL PAIRS ----
pairs_final = match_by_frame_lag(detL, detR, best_lag, MAX_PAIRS_FINAL, tol_frames=MATCH_TOL_FRAMES)
print("Pairs for final (pre-filter):", len(pairs_final))
if len(pairs_final) < 15:
    raise RuntimeError("Final pair çok az. Sol video chessboard'u az görüyor / eşleşme tol düşük.")

# ---- RANSAC PAIR FILTER ----
pairs_clean, kept_scores, all_scores = ransac_filter_pairs(pairs_final)
print(f"RANSAC pair filter: keeping {len(pairs_clean)}/{len(pairs_final)}")

# ---- FINAL CALIBRATION ----
retL, retR, retS, K1, d1, K2, d2, R, T, E, F, img_size = stereo_final(pairs_clean)

print("=== FINAL RESULTS ===")
print("Image size:", img_size)
print("T:", T.ravel())
report_quality(retL, retR, retS, T, kept_scores)

# ---- SAVE ----
np.savez(
    "stereo_calib_final.npz",
    K1=K1, d1=d1, K2=K2, d2=d2,
    R=R, T=T, E=E, F=F,
    img_size=np.array(img_size),
    best_lag=np.array([best_lag], dtype=np.int32),
    sweep_rms=np.array([best_rms], dtype=np.float64),
    ransac_pair_scores=all_scores
)
print("Saved: stereo_calib_final.npz")


LEFT detections (stride=4): 52
LEFT detections (stride=2): 101
RIGHT detections (stride=4): 27
RIGHT detections (stride=2): 50
RIGHT detections (stride=1): 96
Final detections: Left=101 (stride=2), Right=96 (stride=1)
Initial lag estimate (frames): 7
Best lag: 17 frames | Sweep RMS (~20 pairs): 28.1692
Pairs for final (pre-filter): 35
RANSAC pair filter: keeping 28/35
=== FINAL RESULTS ===
Image size: (1280, 720)
T: [ 854.44991976 1365.67636191 2611.9967907 ]
=== QUALITY REPORT ===
RMS L/R/S: 0.353 / 0.373 / 9.329
|T| (same unit as SQUARE_SIZE): 3068.825
Median pair inlier-frac: 1.000
[WARN] Stereo RMS yüksek -> eşleşme/false positive/lag sorunlu olabilir.
[WARN] Baseline aşırı büyük -> yanlış eşleşme/yanlış board parametreleri olabilir.
Saved: stereo_calib_final.npz


In [3]:
import cv2
import numpy as np

# ============================================================
# 2x GoPro Stereo Calibration (ROBUST + WIDE-AWARE)  [PATCHED]
# Key ideas:
# 1) Robust corner detection + strict homography validation
# 2) Pairing by FRAME index with tolerance (nearest frame)
# 3) LAG selection is done by a GLOBAL Fundamental Matrix RANSAC score
#    (much more reliable than stereo RMS for lag sweep)
# 4) Final stereo is solved with FIX_INTRINSIC (intrinsics fixed) to stabilize T
# 5) Pair filtering is done using GLOBAL epipolar residuals (one F for all pairs)
# ============================================================

# ================== INPUTS ==================
LEFT_VIDEO  = "calib7_left.mp4"
RIGHT_VIDEO = "calib7_right.mp4"

PATTERN = (6, 5)          # (cols, rows) inner corners
SQUARE_SIZE_MM = 130.0    # !!! double-check this is correct

# ================== DETECTION ==================
SCALE = 0.5
STRIDE_CANDIDATES = [4, 2, 1]
MIN_DET_TARGET = 80
MIN_DET_ACCEPT = 20
MIN_SEP_DET = 4
MAX_DET = 600

# ================== LAG / PAIRS ==================
LAG_SWEEP_HALF = 60           # frames (increase if videos start far apart)
MAX_PAIRS_SWEEP = 120         # use more for stable global-F scoring
MAX_PAIRS_FINAL = 250
MATCH_TOL_FRAMES = 4          # tolerance for nearest-frame match

# ================== VALIDATION ==================
HOMOGRAPHY_RMSE_MAX = 2.5     # px (2.0 strict; 2.5-3.0 often ok)
BORDER_REJECT_PX = 20

# ================== OPTIMIZATION ==================
SUBPIX_CRIT = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 1e-6)
CALIB_CRIT  = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 80, 1e-7)

# Wide lens intrinsics
INTRINSIC_FLAGS = cv2.CALIB_RATIONAL_MODEL
# Optional (try only if you must):
# INTRINSIC_FLAGS |= cv2.CALIB_THIN_PRISM_MODEL

# Stereo: keep intrinsics fixed (stabilizes baseline / T)
STEREO_FLAGS = cv2.CALIB_FIX_INTRINSIC

# Global-F RANSAC (for lag selection and pair filtering)
GLOBAL_F_RANSAC_THR = 1.0     # px (0.8-2.0)
GLOBAL_F_CONF = 0.999
GLOBAL_F_ITERS = 20000

# Outlier filtering by global epipolar residuals
OUTLIER_KEEP_Q = 0.80         # keep best 80% pairs (by epipolar residual)

# ================== OBJECT POINTS ==================
objp = np.zeros((PATTERN[0] * PATTERN[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:PATTERN[0], 0:PATTERN[1]].T.reshape(-1, 2)
objp *= SQUARE_SIZE_MM


# ================== HELPERS ==================
def preprocess_for_cb(gray):
    """Noise suppress + contrast boost."""
    g = cv2.medianBlur(gray, 3)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    g = clahe.apply(g)
    return g


def _find_corners(gray_small):
    """SB first, then classic fallback."""
    ok, corners = cv2.findChessboardCornersSB(
        gray_small, PATTERN, flags=cv2.CALIB_CB_NORMALIZE_IMAGE
    )
    if ok and corners is not None:
        return corners.astype(np.float32)

    ok, corners = cv2.findChessboardCorners(
        gray_small, PATTERN,
        flags=cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE
    )
    if ok and corners is not None:
        return corners.astype(np.float32)

    return None


def validate_corners_homography(corners, pattern, max_rmse_px=2.5):
    """Reject false positives by homography RMSE against ideal grid."""
    cols, rows = pattern
    N = cols * rows
    if corners is None or len(corners) != N:
        return False, np.inf

    pts = corners.reshape(-1, 2).astype(np.float32)
    grid = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2).astype(np.float32)

    H, _ = cv2.findHomography(grid, pts, method=0)
    if H is None:
        return False, np.inf

    proj = cv2.perspectiveTransform(grid.reshape(-1, 1, 2), H).reshape(-1, 2)
    rmse = float(np.sqrt(np.mean(np.sum((proj - pts) ** 2, axis=1))))
    return (rmse <= max_rmse_px), rmse


def reject_if_near_border(corners, w, h, border=BORDER_REJECT_PX):
    pts = corners.reshape(-1, 2)
    if (pts[:, 0].min() < border or pts[:, 0].max() > (w - border) or
        pts[:, 1].min() < border or pts[:, 1].max() > (h - border)):
        return True
    return False


def find_corners_scaled(gray, scale=SCALE):
    """
    Detect corners on downscaled preprocessed image, map back to original,
    then subpixel refine and validate.
    """
    g = preprocess_for_cb(gray)

    if scale != 1.0:
        small = cv2.resize(g, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    else:
        small = g

    corners = _find_corners(small)
    if corners is None:
        return None

    if scale != 1.0:
        corners = corners / scale

    # refine on preprocessed full-res
    corners_ref = corners.copy()
    cv2.cornerSubPix(g, corners_ref, (11, 11), (-1, -1), SUBPIX_CRIT)

    h, w = gray.shape[:2]
    if reject_if_near_border(corners_ref, w, h):
        return None

    ok, _rmse = validate_corners_homography(corners_ref, PATTERN, HOMOGRAPHY_RMSE_MAX)
    if not ok:
        return None

    return corners_ref


def collect_detections(video_path, stride):
    """
    Returns list of dicts:
      {"frame": idx, "corners": (N,1,2), "size": (w,h), "centroid": (2,)}
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Video açılamadı: {video_path}")

    det = []
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if (frame_idx % stride) != 0:
            frame_idx += 1
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        corners = find_corners_scaled(gray)
        if corners is not None:
            cen = corners.reshape(-1, 2).mean(axis=0)
            h, w = gray.shape[:2]
            det.append({"frame": frame_idx, "corners": corners, "size": (w, h), "centroid": cen})
            if len(det) >= MAX_DET:
                break

        frame_idx += 1

    cap.release()

    # thin out to increase diversity
    filtered = []
    last_f = -10**9
    min_sep_frames = MIN_SEP_DET * stride
    for d in det:
        if d["frame"] - last_f >= min_sep_frames:
            filtered.append(d)
            last_f = d["frame"]

    return filtered


def collect_with_adaptive_stride(video_path, label):
    best = None
    best_stride = None
    for stride in STRIDE_CANDIDATES:
        det = collect_detections(video_path, stride=stride)
        print(f"{label} detections (stride={stride}): {len(det)}")
        if best is None or len(det) > len(best):
            best, best_stride = det, stride
        if len(det) >= MIN_DET_TARGET:
            return det, best_stride
    return best, best_stride


def estimate_lag_frames_from_frames(detL, detR):
    """
    Rough lag init from first/last frames overlap:
    - If videos start far apart, this gives a sane initial guess.
    Then sweep will refine by global-F score.
    """
    fL = np.array([d["frame"] for d in detL], dtype=np.int64)
    fR = np.array([d["frame"] for d in detR], dtype=np.int64)
    # initial guess: align medians (robust)
    return int(np.median(fR) - np.median(fL))


def match_by_frame_lag(detL, detR, lag_frames, max_pairs, tol_frames=MATCH_TOL_FRAMES):
    """
    Pair by frame index:
      frameR ~= frameL + lag_frames
    nearest match within tol_frames.
    Returns list: (cornersL, cornersR, img_size, frameL, frameR)
    """
    mapR = {d["frame"]: d for d in detR}
    framesR = np.array(sorted(mapR.keys()), dtype=np.int64)

    pairs = []
    for dL in detL:
        target = int(dL["frame"] + lag_frames)
        j = int(np.searchsorted(framesR, target))

        candidates = []
        if j < len(framesR):
            candidates.append(int(framesR[j]))
        if j - 1 >= 0:
            candidates.append(int(framesR[j - 1]))
        if not candidates:
            continue

        best_fr = min(candidates, key=lambda fr: abs(fr - target))
        if abs(best_fr - target) > tol_frames:
            continue

        dR = mapR[best_fr]
        if dL["size"] != dR["size"]:
            continue

        pairs.append((dL["corners"], dR["corners"], dL["size"], int(dL["frame"]), int(best_fr)))
        if len(pairs) >= max_pairs:
            break

    return pairs


def build_points(pairs):
    objpoints, imgpointsL, imgpointsR = [], [], []
    for cL, cR, _, _, _ in pairs:
        objpoints.append(objp.copy())
        imgpointsL.append(cL)
        imgpointsR.append(cR)
    return objpoints, imgpointsL, imgpointsR


def _stack_points_for_F(pairs):
    ptsL_all = []
    ptsR_all = []
    for cL, cR, *_ in pairs:
        ptsL_all.append(cL.reshape(-1, 2))
        ptsR_all.append(cR.reshape(-1, 2))
    return np.vstack(ptsL_all).astype(np.float32), np.vstack(ptsR_all).astype(np.float32)


def global_F_score(pairs):
    """
    Score lag candidates by GLOBAL Fundamental matrix inlier ratio.
    Higher = better (more consistent epipolar geometry).
    """
    if len(pairs) < 12:
        return 0.0

    ptsL, ptsR = _stack_points_for_F(pairs)
    F, mask = cv2.findFundamentalMat(
        ptsL, ptsR,
        method=cv2.FM_RANSAC,
        ransacReprojThreshold=GLOBAL_F_RANSAC_THR,
        confidence=GLOBAL_F_CONF,
        maxIters=GLOBAL_F_ITERS
    )
    if F is None or mask is None:
        return 0.0
    return float(mask.sum()) / float(len(mask))


def compute_pair_epipolar_residuals(pairs, F):
    """
    For each pair, compute mean point-to-epipolar-line distance (px)
    using the GLOBAL F (one F for all pairs).
    """
    residuals = []
    for (cL, cR, _, _, _) in pairs:
        ptsL = cL.reshape(-1, 2).astype(np.float64)
        ptsR = cR.reshape(-1, 2).astype(np.float64)

        ones = np.ones((ptsL.shape[0], 1), dtype=np.float64)
        x = np.hstack([ptsL, ones])  # Nx3

        l = (F @ x.T).T  # Nx3 lines in right image
        a, b, c = l[:, 0], l[:, 1], l[:, 2]
        x2, y2 = ptsR[:, 0], ptsR[:, 1]

        dist = np.abs(a * x2 + b * y2 + c) / (np.sqrt(a * a + b * b) + 1e-12)
        residuals.append(float(dist.mean()))
    return np.array(residuals, dtype=np.float64)


def filter_pairs_by_global_F(pairs, keep_q=OUTLIER_KEEP_Q):
    """
    Fit one global F over all points, then drop worst pairs by residual.
    Returns: pairs_kept, F, residuals
    """
    ptsL, ptsR = _stack_points_for_F(pairs)
    F, mask = cv2.findFundamentalMat(
        ptsL, ptsR,
        method=cv2.FM_RANSAC,
        ransacReprojThreshold=GLOBAL_F_RANSAC_THR,
        confidence=GLOBAL_F_CONF,
        maxIters=GLOBAL_F_ITERS
    )
    if F is None:
        return pairs, None, None

    residuals = compute_pair_epipolar_residuals(pairs, F)
    thr = float(np.quantile(residuals, keep_q))
    kept = [p for p, r in zip(pairs, residuals) if r <= thr]

    # ensure minimum
    if len(kept) < 20 and len(pairs) >= 20:
        # relax keep_q
        thr = float(np.quantile(residuals, 0.90))
        kept = [p for p, r in zip(pairs, residuals) if r <= thr]

    return kept, F, residuals


def calibrate_intrinsics(pairs):
    objpoints, imgpointsL, imgpointsR = build_points(pairs)
    img_size = pairs[0][2]

    retL, K1, d1, *_ = cv2.calibrateCamera(
        objpoints, imgpointsL, img_size, None, None,
        flags=INTRINSIC_FLAGS, criteria=CALIB_CRIT
    )
    retR, K2, d2, *_ = cv2.calibrateCamera(
        objpoints, imgpointsR, img_size, None, None,
        flags=INTRINSIC_FLAGS, criteria=CALIB_CRIT
    )
    return float(retL), float(retR), K1, d1, K2, d2, img_size


def stereo_calibrate_fixed_intrinsics(pairs, K1, d1, K2, d2, img_size):
    objpoints, imgpointsL, imgpointsR = build_points(pairs)
    retS, K1o, d1o, K2o, d2o, R, T, E, F = cv2.stereoCalibrate(
        objpoints, imgpointsL, imgpointsR,
        K1, d1, K2, d2,
        img_size,
        criteria=CALIB_CRIT,
        flags=STEREO_FLAGS
    )
    return float(retS), K1o, d1o, K2o, d2o, R, T, E, F


def report_quality(retL, retR, retS, T, pairs, lag, score, residuals=None):
    baseline = float(np.linalg.norm(T.reshape(-1)))
    print("\n=== QUALITY REPORT ===")
    print(f"Pairs used: {len(pairs)}")
    print(f"Best lag (frames): {lag} | global-F score: {score:.3f}")
    print(f"RMS L/R/S: {retL:.3f} / {retR:.3f} / {retS:.3f}")
    print(f"|T| (same unit as SQUARE_SIZE): {baseline:.3f}")

    if residuals is not None and len(residuals):
        print(f"Epipolar residuals (px): median={np.median(residuals):.3f}  p90={np.quantile(residuals,0.90):.3f}")

    if retS > 3.0:
        print("[WARN] Stereo RMS yüksek -> lag/pairing/false-positive sorunu olabilir.")
    if baseline > 1000.0:
        print("[WARN] Baseline aşırı büyük -> lag/pairing veya SQUARE_SIZE_MM yanlış olabilir.")


# ================== RUN ==================
detL, strideL = collect_with_adaptive_stride(LEFT_VIDEO, "LEFT")
detR, strideR = collect_with_adaptive_stride(RIGHT_VIDEO, "RIGHT")

print(f"Final detections: Left={len(detL)} (stride={strideL}), Right={len(detR)} (stride={strideR})")

if len(detL) < MIN_DET_ACCEPT or len(detR) < MIN_DET_ACCEPT:
    raise RuntimeError(
        "Detections çok az. Validation çok sert olabilir.\n"
        "Öneri: HOMOGRAPHY_RMSE_MAX=3.0, BORDER_REJECT_PX=10, SCALE=0.7, STRIDE=1."
    )

# ---- LAG INIT + SWEEP (GLOBAL F SCORE) ----
lag0 = estimate_lag_frames_from_frames(detL, detR)
print("Initial lag guess (frames):", lag0)

best_score = -1.0
best_lag = None

for lag_try in range(lag0 - LAG_SWEEP_HALF, lag0 + LAG_SWEEP_HALF + 1):
    pairs_try = match_by_frame_lag(detL, detR, lag_try, max_pairs=MAX_PAIRS_SWEEP, tol_frames=MATCH_TOL_FRAMES)
    score = global_F_score(pairs_try)
    if score > best_score:
        best_score = score
        best_lag = lag_try

if best_lag is None or best_score <= 0.0:
    raise RuntimeError(
        "Lag sweep başarısız (global-F score düşük).\n"
        "Öneri: LAG_SWEEP_HALF artır (örn 120), MATCH_TOL_FRAMES artır (örn 6), STRIDE=1."
    )

print(f"Best lag by GLOBAL-F: {best_lag} frames | score={best_score:.3f}")

# ---- BUILD FINAL PAIRS ----
pairs_final = match_by_frame_lag(detL, detR, best_lag, max_pairs=MAX_PAIRS_FINAL, tol_frames=MATCH_TOL_FRAMES)
print("Pairs for final (pre-filter):", len(pairs_final))

if len(pairs_final) < 20:
    raise RuntimeError(
        f"Final pair çok az: {len(pairs_final)}.\n"
        "Öneri: MATCH_TOL_FRAMES artır (6), MIN_SEP_DET azalt (2), STRIDE=1."
    )

# ---- FILTER PAIRS by GLOBAL F RESIDUALS ----
pairs_clean, Fg, residuals = filter_pairs_by_global_F(pairs_final, keep_q=OUTLIER_KEEP_Q)
print(f"Global-F pair filter: keeping {len(pairs_clean)}/{len(pairs_final)}")

if len(pairs_clean) < 20:
    print("[WARN] Clean pair sayısı düşük. OUTLIER_KEEP_Q=0.9 deneyebilirsin.")

# ---- INTRINSICS (WIDE-AWARE) ----
retL, retR, K1, d1, K2, d2, img_size = calibrate_intrinsics(pairs_clean)

# ---- STEREO (FIX_INTRINSIC) ----
retS, K1o, d1o, K2o, d2o, R, T, E, F = stereo_calibrate_fixed_intrinsics(
    pairs_clean, K1, d1, K2, d2, img_size
)

print("\n=== FINAL RESULTS ===")
print("Image size:", img_size)
print("T:", T.ravel())

report_quality(retL, retR, retS, T, pairs_clean, best_lag, best_score, residuals=residuals)

# ---- SAVE ----
np.savez(
    "stereo_calib_final.npz",
    K1=K1o, d1=d1o, K2=K2o, d2=d2o,
    R=R, T=T, E=E, F=F,
    F_global=Fg if Fg is not None else F,
    img_size=np.array(img_size),
    best_lag=np.array([best_lag], dtype=np.int32),
    best_globalF_score=np.array([best_score], dtype=np.float64),
    globalF_ransac_thr=np.array([GLOBAL_F_RANSAC_THR], dtype=np.float64),
    match_tol_frames=np.array([MATCH_TOL_FRAMES], dtype=np.int32),
    stride_left=np.array([strideL], dtype=np.int32),
    stride_right=np.array([strideR], dtype=np.int32),
    pair_epipolar_residuals=residuals if residuals is not None else np.array([], dtype=np.float64),
)

print("Saved: stereo_calib_final.npz")


LEFT detections (stride=4): 52
LEFT detections (stride=2): 101
RIGHT detections (stride=4): 27
RIGHT detections (stride=2): 50
RIGHT detections (stride=1): 96
Final detections: Left=101 (stride=2), Right=96 (stride=1)
Initial lag guess (frames): -56
Best lag by GLOBAL-F: 1 frames | score=0.526
Pairs for final (pre-filter): 49
Global-F pair filter: keeping 39/49

=== FINAL RESULTS ===
Image size: (1280, 720)
T: [2362.82039859 2180.53553982 1604.80197925]

=== QUALITY REPORT ===
Pairs used: 39
Best lag (frames): 1 | global-F score: 0.526
RMS L/R/S: 0.361 / 0.365 / 14.579
|T| (same unit as SQUARE_SIZE): 3593.473
Epipolar residuals (px): median=0.714  p90=93.112
[WARN] Stereo RMS yüksek -> lag/pairing/false-positive sorunu olabilir.
[WARN] Baseline aşırı büyük -> lag/pairing veya SQUARE_SIZE_MM yanlış olabilir.
Saved: stereo_calib_final.npz


In [4]:
import os
import csv
import cv2
import numpy as np

LEFT_VIDEO  = "calib7_left.mp4"
RIGHT_VIDEO = "calib7_right.mp4"

PATTERN = (6, 5)  # inner corners (cols, rows)

# ======= Ayarlar =======
SCALE = 0.6           # detection için küçültme
STRIDE = 1            # her frame dene (gerekirse 2-3 yap)
MAX_DET = 400         # en fazla kaç detection toplasın
MIN_SEP = 1           # detectionlar arası minimum frame farkı
PAIR_TOL = 2          # lag ile eşleştirirken +/- tolerans
EXPORT_N = 80         # kaç eşleşme kaydedilsin

# Eğer best_lag biliyorsan buraya yaz:
BEST_LAG = None       # örn: 667
SWEEP_HALF = 1200     # BEST_LAG None ise, lag taraması aralığı

OUT_DIR = "out_debug"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "pairs"), exist_ok=True)

subpix_crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)


def detect_chess(video_path, stride=1):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError("Video açılamadı: " + video_path)

    det = []
    i = 0
    last = -10**9

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if i % stride == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            if SCALE != 1.0:
                gs = cv2.resize(gray, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_AREA)
            else:
                gs = gray

            ok, corners = cv2.findChessboardCorners(
                gs, PATTERN,
                flags=cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE
            )

            if ok:
                corners = corners.astype(np.float32)
                if SCALE != 1.0:
                    corners /= SCALE

                # subpixel refine (full-res gray)
                cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), subpix_crit)

                if i - last >= MIN_SEP:
                    det.append((i, corners))
                    last = i

                if len(det) >= MAX_DET:
                    break

        i += 1

    cap.release()
    return det


def get_frame(video_path, idx):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None


def match_count(detL, detR_map, detR_frames, lag):
    cnt = 0
    for fL, _ in detL:
        target = fL + lag
        if target in detR_map:
            cnt += 1
        else:
            j = int(np.argmin(np.abs(detR_frames - target)))
            if abs(int(detR_frames[j]) - target) <= PAIR_TOL:
                cnt += 1
    return cnt


def main():
    print("Detecting chessboards...")
    detL = detect_chess(LEFT_VIDEO, STRIDE)
    detR = detect_chess(RIGHT_VIDEO, STRIDE)
    print("Detections L/R:", len(detL), len(detR))

    detR_map = {f: c for f, c in detR}
    detR_frames = np.array([f for f, _ in detR], dtype=int)

    if len(detL) < 10 or len(detR) < 10:
        print("Çok az detection var. STRIDE=1, SCALE=0.8 deneyebilirsin.")
        return

    # ---- Lag seçimi ----
    if BEST_LAG is None:
        guess = int(np.median(detR_frames) - np.median([f for f, _ in detL]))
        best_lag, best_c = None, -1

        print("Sweeping lag around:", guess, "+/-", SWEEP_HALF)
        for lag in range(guess - SWEEP_HALF, guess + SWEEP_HALF + 1):
            c = match_count(detL, detR_map, detR_frames, lag)
            if c > best_c:
                best_c = c
                best_lag = lag

        lag = best_lag
    else:
        lag = int(BEST_LAG)

    print("Chosen lag:", lag, "matches:", match_count(detL, detR_map, detR_frames, lag))

    # ---- Export ----
    rows = []
    for fL, cL in detL:
        if len(rows) >= EXPORT_N:
            break

        target = fL + lag
        fR = None

        if target in detR_map:
            fR = target
        else:
            j = int(np.argmin(np.abs(detR_frames - target)))
            cand = int(detR_frames[j])
            if abs(cand - target) <= PAIR_TOL:
                fR = cand

        if fR is None:
            continue

        frameL = get_frame(LEFT_VIDEO, fL)
        frameR = get_frame(RIGHT_VIDEO, fR)
        if frameL is None or frameR is None:
            continue

        # Draw overlays
        cv2.putText(frameL, f"L {fL}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(frameR, f"R {fR}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.drawChessboardCorners(frameL, PATTERN, cL, True)
        cv2.drawChessboardCorners(frameR, PATTERN, detR_map[fR], True)

        # Side-by-side
        h = max(frameL.shape[0], frameR.shape[0])
        w = frameL.shape[1] + frameR.shape[1]
        canvas = np.zeros((h, w, 3), dtype=np.uint8)
        canvas[:frameL.shape[0], :frameL.shape[1]] = frameL
        canvas[:frameR.shape[0], frameL.shape[1]:frameL.shape[1] + frameR.shape[1]] = frameR

        out_path = os.path.join(OUT_DIR, "pairs", f"pair_L{fL}_R{fR}.jpg")
        cv2.imwrite(out_path, canvas)

        rows.append([len(rows), fL, fR, out_path])

    csv_path = os.path.join(OUT_DIR, "pairs.csv")
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["idx", "frameL", "frameR", "file"])
        w.writerows(rows)

    print(f"Exported {len(rows)} pairs.")
    print("Folder:", os.path.abspath(OUT_DIR))
    print("CSV:", os.path.abspath(csv_path))


if __name__ == "__main__":
    main()


Detecting chessboards...
Detections L/R: 400 400
Sweeping lag around: -3 +/- 1200
Chosen lag: -1 matches: 390
Exported 80 pairs.
Folder: C:\Users\edanu\OneDrive - Koc Universitesi\Masaüstü\Koc_Calismalar\AUTOTRACKING_dennis\Autotracking Version 2\out_debug
CSV: C:\Users\edanu\OneDrive - Koc Universitesi\Masaüstü\Koc_Calismalar\AUTOTRACKING_dennis\Autotracking Version 2\out_debug\pairs.csv


In [5]:
import cv2
import numpy as np

LEFT_VIDEO  = "calib7_left.mp4"
RIGHT_VIDEO = "calib7_right.mp4"

PATTERN = (6, 5)
SQUARE_SIZE_MM = 130.0

# FAST SETTINGS
SCALE = 0.5
STRIDE = 2
MAX_DET = 250
MIN_SEP = 2

BEST_LAG = -1     # sende bu net gibi
PAIR_TOL = 2
MAX_PAIRS_CALIB = 80
MIN_PAIRS_CALIB = 20

INTRINSIC_FLAGS = cv2.CALIB_RATIONAL_MODEL
STEREO_FLAGS = cv2.CALIB_FIX_INTRINSIC

subpix_crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-6)
calib_crit  = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 60, 1e-7)

def make_objp():
    objp = np.zeros((PATTERN[0]*PATTERN[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:PATTERN[0], 0:PATTERN[1]].T.reshape(-1, 2)
    objp *= float(SQUARE_SIZE_MM)
    return objp

def detect(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError("Video açılamadı: " + video_path)

    det = []
    i = 0
    last = -10**9
    img_size = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if i % STRIDE == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            if img_size is None:
                img_size = (gray.shape[1], gray.shape[0])

            gs = cv2.resize(gray, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_AREA) if SCALE != 1.0 else gray

            ok, corners = cv2.findChessboardCornersSB(gs, PATTERN, flags=cv2.CALIB_CB_NORMALIZE_IMAGE)
            if not ok:
                ok, corners = cv2.findChessboardCorners(gs, PATTERN,
                    flags=cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE)

            if ok and corners is not None:
                corners = corners.astype(np.float32)
                if SCALE != 1.0:
                    corners /= float(SCALE)
                cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), subpix_crit)

                if i - last >= MIN_SEP:
                    det.append((i, corners, img_size))
                    last = i

                if len(det) >= MAX_DET:
                    break

        i += 1

    cap.release()
    return det

def build_pairs(detL, detR):
    detR_map = {f: (c, sz) for f, c, sz in detR}
    detR_frames = np.array(sorted(detR_map.keys()), dtype=int)

    pairs = []
    lag = int(BEST_LAG)

    for fL, cL, szL in detL:
        target = fL + lag
        fR = None

        if target in detR_map:
            fR = target
        else:
            j = int(np.argmin(np.abs(detR_frames - target)))
            cand = int(detR_frames[j])
            if abs(cand - target) <= PAIR_TOL:
                fR = cand

        if fR is None:
            continue

        cR, szR = detR_map[fR]
        if szL != szR:
            continue

        pairs.append((cL, cR, szL))
        if len(pairs) >= MAX_PAIRS_CALIB:
            break

    return pairs

def stereo_calib(pairs):
    objp = make_objp()
    objpoints, imgL, imgR = [], [], []
    img_size = pairs[0][2]

    for cL, cR, _ in pairs:
        objpoints.append(objp.copy())
        imgL.append(cL)
        imgR.append(cR)

    retL, K1, d1, _, _ = cv2.calibrateCamera(objpoints, imgL, img_size, None, None,
                                            flags=INTRINSIC_FLAGS, criteria=calib_crit)
    retR, K2, d2, _, _ = cv2.calibrateCamera(objpoints, imgR, img_size, None, None,
                                            flags=INTRINSIC_FLAGS, criteria=calib_crit)

    retS, K1, d1, K2, d2, R, T, E, F = cv2.stereoCalibrate(
        objpoints, imgL, imgR, K1, d1, K2, d2, img_size,
        flags=STEREO_FLAGS, criteria=calib_crit
    )
    return retL, retR, retS, K1, d1, K2, d2, R, T, E, F, img_size

def main():
    detL = detect(LEFT_VIDEO)
    detR = detect(RIGHT_VIDEO)
    print("Detections L/R:", len(detL), len(detR))

    pairs = build_pairs(detL, detR)
    print("Pairs:", len(pairs))
    if len(pairs) < MIN_PAIRS_CALIB:
        raise RuntimeError("Pair az. STRIDE=1 yap veya MAX_DET artır ya da PAIR_TOL=3 dene.")

    retL, retR, retS, K1, d1, K2, d2, R, T, E, F, img_size = stereo_calib(pairs)
    print("Image size:", img_size)
    print("RMS L/R/S:", retL, retR, retS)
    print("T:", T.ravel(), "|T|:", float(np.linalg.norm(T.ravel())))

    np.savez("stereo_calib_final.npz",
             K1=K1, d1=d1, K2=K2, d2=d2, R=R, T=T, E=E, F=F,
             img_size=np.array(img_size),
             best_lag=np.array([BEST_LAG], dtype=np.int32),
             used_pairs=np.array([len(pairs)], dtype=np.int32),
             square_size_mm=np.array([SQUARE_SIZE_MM], dtype=np.float64))
    print("Saved: stereo_calib_final.npz")

if __name__ == "__main__":
    main()


Detections L/R: 250 250
Pairs: 80
Image size: (1280, 720)
RMS L/R/S: 0.44195258858410646 0.6714298551207386 14.956162523019055
T: [-2535.60932051  1298.27864786  1541.12268745] |T|: 3238.811697449946
Saved: stereo_calib_final.npz


In [6]:
import os, csv, cv2, numpy as np

LEFT_VIDEO  = "calib7_left.mp4"
RIGHT_VIDEO = "calib7_right.mp4"
OUT_DIR = "sync_check"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "pairs"), exist_ok=True)

# senin bulduğun lag
LAG0 = -1
SEARCH = 2          # lag0 +/- SEARCH
N_EXPORT = 80
STRIDE = 3          # hız için
DOWNSCALE = 0.5     # hız için

def get_frame(cap, idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ret, fr = cap.read()
    return fr if ret else None

def sim_score(a, b):
    # hızlı: grayscale + downscale + normalized correlation
    a = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY)
    b = cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)
    if DOWNSCALE != 1.0:
        a = cv2.resize(a, None, fx=DOWNSCALE, fy=DOWNSCALE, interpolation=cv2.INTER_AREA)
        b = cv2.resize(b, None, fx=DOWNSCALE, fy=DOWNSCALE, interpolation=cv2.INTER_AREA)
    a = a.astype(np.float32); b = b.astype(np.float32)
    a -= a.mean(); b -= b.mean()
    denom = (a.std()*b.std() + 1e-6)
    return float((a*b).mean() / denom)

def main():
    capL = cv2.VideoCapture(LEFT_VIDEO)
    capR = cv2.VideoCapture(RIGHT_VIDEO)
    if not capL.isOpened() or not capR.isOpened():
        raise RuntimeError("Video açılamadı")

    rows = []
    i = 0
    exported = 0

    while exported < N_EXPORT:
        frL = get_frame(capL, i)
        if frL is None:
            break

        if i % STRIDE != 0:
            i += 1
            continue

        best = None
        best_score = -1e9
        best_j = None

        # aday sağ frameler: i + lag0 + delta
        for d in range(-SEARCH, SEARCH+1):
            j = i + LAG0 + d
            if j < 0:
                continue
            frR = get_frame(capR, j)
            if frR is None:
                continue
            s = sim_score(frL, frR)
            if s > best_score:
                best_score = s
                best = frR
                best_j = j

        if best is not None:
            canvas = np.hstack([frL, best])
            cv2.putText(canvas, f"L {i}", (20,40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
            cv2.putText(canvas, f"R {best_j}  score={best_score:.3f}", (frL.shape[1]+20,40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

            outp = os.path.join(OUT_DIR, "pairs", f"pair_L{i}_R{best_j}.jpg")
            cv2.imwrite(outp, canvas)
            rows.append([exported, i, best_j, best_j - i, best_score, outp])
            exported += 1

        i += 1

    capL.release(); capR.release()

    csv_path = os.path.join(OUT_DIR, "sync_report.csv")
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["idx","frameL","frameR","delta(R-L)","score","file"])
        w.writerows(rows)

    deltas = [r[3] for r in rows]
    if deltas:
        print("Export:", exported)
        print("Delta stats: min/med/max =", min(deltas), int(np.median(deltas)), max(deltas))
        print("Unique deltas:", sorted(set(deltas))[:20], "... (total", len(set(deltas)), ")")
    print("Folder:", os.path.abspath(OUT_DIR))
    print("CSV:", os.path.abspath(csv_path))

if __name__ == "__main__":
    main()


Export: 80
Delta stats: min/med/max = -3 0 1
Unique deltas: [-3, -2, -1, 0, 1] ... (total 5 )
Folder: C:\Users\edanu\OneDrive - Koc Universitesi\Masaüstü\Koc_Calismalar\AUTOTRACKING_dennis\Autotracking Version 2\sync_check
CSV: C:\Users\edanu\OneDrive - Koc Universitesi\Masaüstü\Koc_Calismalar\AUTOTRACKING_dennis\Autotracking Version 2\sync_check\sync_report.csv


In [1]:
import cv2
import numpy as np

# ============================================================
# 2x GoPro Stereo Calibration (AUTO-PAIR BY SIMILARITY)
# - Detect chessboard frames (stride + downscale)
# - For each detection, compute a compact frame "signature"
# - Estimate lag (centroid motion)
# - Lag sweep: for each L frame, search R frames in +/- SEARCH_WIN
#   and pick the most similar one (cosine similarity of signatures)
# - Stereo RMS to pick best lag
# - Epipolar outlier cleanup
# - Save npz
# ============================================================

# ================== INPUTS ==================
LEFT_VIDEO  = "calib7_left.mp4"
RIGHT_VIDEO = "calib7_right.mp4"

PATTERN = (6, 5)         # (cols, rows) inner corners
SQUARE_SIZE_MM = 130.0

# ================== SPEED / ROBUSTNESS ==================
SCALE = 0.5              # downscale for chessboard detection
STRIDE_DET = 2           # try 2; if detections low -> 1
MAX_DET = 400            # max detections per side
MIN_SEP_DET = 2          # min frame gap between detections (after stride)

# Signature (similarity) settings
SIG_W, SIG_H = 96, 54    # small size for signature (speed)
SEARCH_WIN = 3           # +/- frames around target to search for best match (2-4 good)
MAX_PAIRS_SWEEP = 20
MAX_PAIRS_FINAL = 120

# Lag sweep
LAG_SWEEP_HALF = 60      # search lag in [lag0-.., lag0+..]
MAX_LAG_FRAMES = 2000

# Calibration
INTRINSIC_FLAGS = cv2.CALIB_RATIONAL_MODEL  # good for wide lenses; set 0 if you want simpler
stereo_flags_sweep = cv2.CALIB_FIX_INTRINSIC
stereo_flags_final = cv2.CALIB_USE_INTRINSIC_GUESS
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 60, 1e-6)

# Outlier cleanup
OUTLIER_DROP_Q = 0.70    # drop worst 30%

# ================== OBJECT POINTS ==================
objp = np.zeros((PATTERN[0]*PATTERN[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:PATTERN[0], 0:PATTERN[1]].T.reshape(-1, 2)
objp *= float(SQUARE_SIZE_MM)

subpix_crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-6)


# ================== HELPERS ==================
def make_signature(gray):
    """
    Compact signature for similarity:
    - resize to (SIG_W, SIG_H)
    - normalize (zero-mean, unit-std)
    - flatten
    """
    small = cv2.resize(gray, (SIG_W, SIG_H), interpolation=cv2.INTER_AREA)
    v = small.astype(np.float32).reshape(-1)
    v -= v.mean()
    s = v.std() + 1e-6
    v /= s
    return v

def cosine_sim(a, b):
    # a,b already normalized-ish; still safe:
    na = np.linalg.norm(a) + 1e-8
    nb = np.linalg.norm(b) + 1e-8
    return float(np.dot(a, b) / (na * nb))

def find_corners_scaled(gray):
    if SCALE != 1.0:
        small = cv2.resize(gray, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_AREA)
    else:
        small = gray

    ok, corners = cv2.findChessboardCornersSB(
        small, PATTERN, flags=cv2.CALIB_CB_NORMALIZE_IMAGE
    )

    if (not ok) or corners is None:
        ok, corners = cv2.findChessboardCorners(
            small, PATTERN,
            flags=cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE
        )

    if (not ok) or corners is None:
        return None

    corners = corners.astype(np.float32)
    if SCALE != 1.0:
        corners /= float(SCALE)

    # subpix on full-res gray
    cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), subpix_crit)
    return corners

def collect_detections(video_path):
    """
    Returns list of dict:
      {"frame": i, "corners": corners, "img_size": (w,h), "centroid": (2,), "sig": (D,)}
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Video açılamadı: {video_path}")

    det = []
    i = 0
    last_kept = -10**9
    img_size = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if (i % STRIDE_DET) == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            if img_size is None:
                img_size = (gray.shape[1], gray.shape[0])

            corners = find_corners_scaled(gray)
            if corners is not None:
                if i - last_kept >= MIN_SEP_DET:
                    cen = corners.reshape(-1, 2).mean(axis=0)
                    sig = make_signature(gray)
                    det.append({
                        "frame": i,
                        "corners": corners,
                        "img_size": img_size,
                        "centroid": cen,
                        "sig": sig
                    })
                    last_kept = i

                if len(det) >= MAX_DET:
                    break

        i += 1

    cap.release()
    return det

def _fill_nan_xy(X):
    X = X.copy()
    for j in range(2):
        x = X[:, j]
        if np.isnan(x).all():
            return None
        for i in range(1, len(x)):
            if np.isnan(x[i]) and not np.isnan(x[i-1]):
                x[i] = x[i-1]
        for i in range(len(x)-2, -1, -1):
            if np.isnan(x[i]) and not np.isnan(x[i+1]):
                x[i] = x[i+1]
        X[:, j] = x
    return X

def estimate_lag_frames(detL, detR):
    """
    Coarse lag estimate using centroid motion signature (like before).
    """
    if len(detL) < 5 or len(detR) < 5:
        raise RuntimeError("Lag tahmini için detection az (>=5 önerilir).")

    fL = np.array([d["frame"] for d in detL], dtype=np.int32)
    fR = np.array([d["frame"] for d in detR], dtype=np.int32)
    cL = np.array([d["centroid"] for d in detL], dtype=np.float64)
    cR = np.array([d["centroid"] for d in detR], dtype=np.float64)

    fmin = int(min(fL.min(), fR.min()))
    fmax = int(max(fL.max(), fR.max()))
    n = fmax - fmin + 1
    if n < 30:
        return int(np.median(fR) - np.median(fL))

    Xl = np.full((n, 2), np.nan, dtype=np.float64)
    Xr = np.full((n, 2), np.nan, dtype=np.float64)
    Xl[fL - fmin] = cL
    Xr[fR - fmin] = cR
    Xl = _fill_nan_xy(Xl)
    Xr = _fill_nan_xy(Xr)
    if Xl is None or Xr is None:
        return int(np.median(fR) - np.median(fL))

    sigL = np.linalg.norm(np.diff(Xl, axis=0), axis=1)
    sigR = np.linalg.norm(np.diff(Xr, axis=0), axis=1)
    sigL = (sigL - sigL.mean()) / (sigL.std() + 1e-9)
    sigR = (sigR - sigR.mean()) / (sigR.std() + 1e-9)

    corr = np.correlate(sigR, sigL, mode="full")
    lag = int(np.argmax(corr) - (len(sigL) - 1))
    lag = int(np.clip(lag, -MAX_LAG_FRAMES, MAX_LAG_FRAMES))
    return lag

def build_pairs_by_similarity(detL, detR, lag_frames, max_pairs):
    """
    For each left detection at frame fL:
      target = fL + lag_frames
      candidates in R: frames within [target-SEARCH_WIN, target+SEARCH_WIN]
      pick candidate with maximum cosine similarity of signatures.

    Also enforces 1-to-1 use of R frames (greedy) to avoid duplicates.
    """
    r_by_frame = {d["frame"]: d for d in detR}
    r_frames = np.array(sorted(r_by_frame.keys()), dtype=np.int32)

    usedR = set()
    pairs = []
    img_size = detL[0]["img_size"]

    for dL in detL:
        fL = int(dL["frame"])
        target = fL + int(lag_frames)

        # get indices near target efficiently
        j0 = int(np.searchsorted(r_frames, target - SEARCH_WIN))
        j1 = int(np.searchsorted(r_frames, target + SEARCH_WIN, side="right"))

        best = None
        best_score = -1e9

        for j in range(j0, j1):
            if j < 0 or j >= len(r_frames):
                continue
            fR = int(r_frames[j])
            if fR in usedR:
                continue
            dR = r_by_frame[fR]
            if dR["img_size"] != img_size:
                continue

            score = cosine_sim(dL["sig"], dR["sig"])
            if score > best_score:
                best_score = score
                best = dR

        if best is None:
            continue

        usedR.add(int(best["frame"]))
        pairs.append((dL["corners"], best["corners"], img_size, fL, int(best["frame"]), best_score))

        if len(pairs) >= max_pairs:
            break

    return pairs

def build_points(pairs):
    objpoints, imgL, imgR = [], [], []
    for cL, cR, _, _, _, _ in pairs:
        objpoints.append(objp.copy())
        imgL.append(cL)
        imgR.append(cR)
    return objpoints, imgL, imgR

def stereo_rms_only(pairs):
    if len(pairs) < 12:
        return np.inf
    objpoints, imgL, imgR = build_points(pairs)
    img_size = pairs[0][2]

    _, K1, d1, *_ = cv2.calibrateCamera(objpoints, imgL, img_size, None, None,
                                        flags=INTRINSIC_FLAGS, criteria=criteria)
    _, K2, d2, *_ = cv2.calibrateCamera(objpoints, imgR, img_size, None, None,
                                        flags=INTRINSIC_FLAGS, criteria=criteria)

    retS, *_ = cv2.stereoCalibrate(objpoints, imgL, imgR,
                                   K1, d1, K2, d2, img_size,
                                   flags=stereo_flags_sweep, criteria=criteria)
    return float(retS)

def stereo_final(pairs):
    if len(pairs) < 15:
        raise RuntimeError(f"Final stereo için pair az: {len(pairs)} (>=15 önerilir).")
    objpoints, imgL, imgR = build_points(pairs)
    img_size = pairs[0][2]

    retL, K1, d1, *_ = cv2.calibrateCamera(objpoints, imgL, img_size, None, None,
                                           flags=INTRINSIC_FLAGS, criteria=criteria)
    retR, K2, d2, *_ = cv2.calibrateCamera(objpoints, imgR, img_size, None, None,
                                           flags=INTRINSIC_FLAGS, criteria=criteria)

    retS, K1, d1, K2, d2, R, T, E, F = cv2.stereoCalibrate(
        objpoints, imgL, imgR,
        K1, d1, K2, d2, img_size,
        flags=stereo_flags_final, criteria=criteria
    )
    return float(retL), float(retR), float(retS), K1, d1, K2, d2, R, T, E, F, img_size

def epipolar_errors(pairs, F):
    errs = []
    for cL, cR, _, _, _, _ in pairs:
        ptsL = cL.reshape(-1, 2)
        ptsR = cR.reshape(-1, 2)
        ones = np.ones((ptsL.shape[0], 1), dtype=np.float64)
        x = np.hstack([ptsL.astype(np.float64), ones])  # Nx3
        l = (F @ x.T).T
        a, b, c = l[:, 0], l[:, 1], l[:, 2]
        x2 = ptsR[:, 0].astype(np.float64)
        y2 = ptsR[:, 1].astype(np.float64)
        dist = np.abs(a * x2 + b * y2 + c) / (np.sqrt(a*a + b*b) + 1e-12)
        errs.append(float(dist.mean()))
    return np.array(errs, dtype=np.float64)


# ================== RUN ==================
print("Collecting detections...")
detL = collect_detections(LEFT_VIDEO)
detR = collect_detections(RIGHT_VIDEO)

print(f"Detections: Left={len(detL)}, Right={len(detR)}")
print("Left frames (first 10):", [d["frame"] for d in detL[:10]])
print("Right frames (first 10):", [d["frame"] for d in detR[:10]])

if len(detL) < 12 or len(detR) < 12:
    raise RuntimeError("Detections çok az. STRIDE_DET=1 yap, SCALE=0.6-0.8 dene, MAX_DET artır.")

lag0 = estimate_lag_frames(detL, detR)
print("Initial lag estimate (frames):", lag0)

# ---- LAG SWEEP with similarity pairing ----
best_rms = 1e9
best_lag = None
best_pairs_for_sweep = None

for lag_try in range(lag0 - LAG_SWEEP_HALF, lag0 + LAG_SWEEP_HALF + 1):
    pairs_try = build_pairs_by_similarity(detL, detR, lag_try, MAX_PAIRS_SWEEP)
    if len(pairs_try) < 12:
        continue
    rms = stereo_rms_only(pairs_try)
    if rms < best_rms:
        best_rms = rms
        best_lag = lag_try
        best_pairs_for_sweep = pairs_try

if best_lag is None:
    raise RuntimeError("Lag sweep başarısız. LAG_SWEEP_HALF artır (örn 120) veya SEARCH_WIN=4 dene.")

print(f"Best lag: {best_lag} | Sweep RMS (~{MAX_PAIRS_SWEEP} pairs): {best_rms:.4f}")

# ---- FINAL PAIRS (similarity pairing) ----
pairs_final = build_pairs_by_similarity(detL, detR, best_lag, MAX_PAIRS_FINAL)
print("Pairs for final:", len(pairs_final))

if len(pairs_final) < 15:
    raise RuntimeError("Final pair az. STRIDE_DET=1 yap veya MAX_DET artır veya SEARCH_WIN=4 dene.")

# ---- FINAL CALIB (pre-clean) ----
retL, retR, retS, K1, d1, K2, d2, R, T, E, F, img_size = stereo_final(pairs_final)
print("=== PRE-CLEAN RESULTS ===")
print("Image size:", img_size)
print("RMS L/R/S:", retL, retR, retS)
print("T (mm):", T.ravel(), "|T|:", float(np.linalg.norm(T.ravel())))

# ---- EPIPOLAR OUTLIER CLEAN ----
errs = epipolar_errors(pairs_final, F)
thr = np.quantile(errs, OUTLIER_DROP_Q)
pairs_clean = [p for p, e in zip(pairs_final, errs) if e <= thr]
print(f"Epipolar cleaning: keeping {len(pairs_clean)}/{len(pairs_final)} (thr={thr:.3f} px)")

# ---- FINAL CALIB (clean) ----
retL2, retR2, retS2, K1c, d1c, K2c, d2c, Rc, Tc, Ec, Fc, _ = stereo_final(pairs_clean)
print("=== FINAL CLEAN RESULTS ===")
print("RMS L/R/S:", retL2, retR2, retS2)
print("T (mm):", Tc.ravel(), "|T|:", float(np.linalg.norm(Tc.ravel())))

# ---- SAVE ----
np.savez(
    "stereo_calib_final.npz",
    K1=K1c, d1=d1c, K2=K2c, d2=d2c,
    R=Rc, T=Tc, E=Ec, F=Fc,
    img_size=np.array(img_size),
    best_lag=np.array([best_lag], dtype=np.int32),
    sweep_rms=np.array([best_rms], dtype=np.float64),
    stride_det=np.array([STRIDE_DET], dtype=np.int32),
    scale=np.array([SCALE], dtype=np.float64),
    search_win=np.array([SEARCH_WIN], dtype=np.int32),
    used_pairs=np.array([len(pairs_clean)], dtype=np.int32),
    square_size_mm=np.array([SQUARE_SIZE_MM], dtype=np.float64),
    epipolar_errs=errs
)
print("Saved: stereo_calib_final.npz")


Detections: Left=400, Right=400
Left frames (first 10): [82, 84, 86, 92, 94, 96, 98, 100, 102, 104]
Right frames (first 10): [82, 88, 92, 98, 100, 102, 104, 106, 108, 110]
Initial lag estimate (frames): 0
Best lag: 13 | Sweep RMS (~20 pairs): 6.6229
Pairs for final: 120
=== PRE-CLEAN RESULTS ===
Image size: (1280, 720)
RMS L/R/S: 0.44214583701630905 0.4058945403894412 23.58941636236027
T (mm): [-2852.80137974  2925.93698805  1122.25382089] |T|: 4237.810355448222
Epipolar cleaning: keeping 84/120 (thr=314.882 px)
=== FINAL CLEAN RESULTS ===
RMS L/R/S: 0.4444765760874324 0.42018486882322836 9.239685183679441
T (mm): [-4561.01170426  2825.16856948   162.13247744] |T|: 5367.559236056738
Saved: stereo_calib_final.npz


In [2]:
import os
import csv
import cv2
import numpy as np

# ============================================================
# 2x GoPro Stereo Calibration (HyperSmooth-ROBUST, AUTO-PAIR)
#
# Amaç:
# - HyperSmooth / lens-warp gibi stabilizasyonların etkisini KODDA azaltmak
#   (tam geri alamazsın; ama daha stabil kalibrasyon çıkarırsın)
#
# Neler var?
# 1) Merkez ROI (crop) ile corner arama  -> kenar warp etkisini azaltır
# 2) Preprocess (median + CLAHE)         -> low-contrast + noise durumunda daha iyi
# 3) SB corner -> fallback classic -> subpix
# 4) Sıkı validasyon:
#    - border reject
#    - homography grid RMSE (false positive shield)
# 5) Auto-pair by similarity:
#    - her sol frame için sağda target±SEARCH_WIN arar
#    - ROI signature cosine similarity ile en benzeri seçer
#    - 1-to-1 (R frame reuse yok)
# 6) Lag sweep (coarse lag0 + sweep)     -> en iyi lag stereo RMS ile seçilir
# 7) Epipolar outlier cleanup
# 8) Debug export:
#    - eşleşen frame çiftlerini yan yana JPG kaydeder
#    - pairs.csv oluşturur
# 9) Kaydeder: stereo_calib_final.npz
#
# Not:
# - SQUARE_SIZE_MM: sadece T'nin birimini mm yapar (ölçek).
#   Stereo RMS / R / F gibi kalite metrikleri bundan bağımsız.
# ============================================================

# ================== INPUTS ==================
LEFT_VIDEO  = "calib7_left.mp4"
RIGHT_VIDEO = "calib7_right.mp4"

PATTERN = (6, 5)         # (cols, rows) inner corners
SQUARE_SIZE_MM = 130.0   # Kare kenarı (mm). Sadece ölçek (T'nin birimi) için.

# ================== DETECTION / SPEED ==================
SCALE = 0.5              # chessboard detection downscale
STRIDE_DET = 2           # 2 iyi; det azsa 1 yap
MAX_DET = 400            # max detections per side
MIN_SEP_DET = 2          # min frame gap between detections (after stride)

# ---- HyperSmooth robust trick: center crop ----
CROP_FRAC = 0.70         # merkezin %70'ini kullan (0.6-0.8 arası dene)

# ================== VALIDATION (FALSE POSITIVE SHIELD) ==================
BORDER_REJECT_PX = 40    # frame/ROI sınırına yakınsa at (warp en çok kenarda)
HOMOGRAPHY_RMSE_MAX = 2.0  # px (2.0-2.5 arası)

# ================== SIGNATURE (PAIRING) ==================
SIG_W, SIG_H = 96, 54    # signature boyutu (hız)
SEARCH_WIN = 3           # target +/- kaç frame ara (2-4)
MAX_PAIRS_SWEEP = 20
MAX_PAIRS_FINAL = 140    # biraz yüksek tut -> outlier sonrası kalsın

# ================== LAG SWEEP ==================
LAG_SWEEP_HALF = 40      # lag0 +/- kaç frame taransın (0 çıkıyorsa 15-25 yeter)
MAX_LAG_FRAMES = 2000

# ================== CALIBRATION ==================
# HyperSmooth varken distortion'ı aşırı serbest bırakma:
# RATIONAL_MODEL bazen "warp'ı distortion sanıp" patlatabiliyor.
# Bu yüzden default model (0) daha stabil olabiliyor.
USE_RATIONAL_MODEL = False

INTRINSIC_FLAGS = 0 if not USE_RATIONAL_MODEL else cv2.CALIB_RATIONAL_MODEL

stereo_flags_sweep = cv2.CALIB_FIX_INTRINSIC
stereo_flags_final = cv2.CALIB_USE_INTRINSIC_GUESS

subpix_crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 1e-6)
criteria    = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 80, 1e-7)

# ================== OUTLIER CLEANUP ==================
OUTLIER_DROP_Q = 0.70    # worst 30% drop

# ================== DEBUG EXPORT ==================
DEBUG_DIR = "stereo_debug"
EXPORT_PAIRS_JPG = True
EXPORT_N = 80            # kaç çift görüntü export edilsin
os.makedirs(DEBUG_DIR, exist_ok=True)
os.makedirs(os.path.join(DEBUG_DIR, "pairs"), exist_ok=True)

# ================== OBJECT POINTS ==================
objp = np.zeros((PATTERN[0]*PATTERN[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:PATTERN[0], 0:PATTERN[1]].T.reshape(-1, 2)
objp *= float(SQUARE_SIZE_MM)


# ============================================================
# Helpers
# ============================================================
def preprocess(gray):
    g = cv2.medianBlur(gray, 3)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    g = clahe.apply(g)
    return g

def center_crop(gray, frac=CROP_FRAC):
    h, w = gray.shape[:2]
    nw, nh = int(w * frac), int(h * frac)
    x0 = (w - nw) // 2
    y0 = (h - nh) // 2
    return gray[y0:y0+nh, x0:x0+nw], (x0, y0, nw, nh)

def uncrop_corners(corners, crop_info):
    x0, y0, _, _ = crop_info
    c = corners.copy()
    c[:, 0, 0] += float(x0)
    c[:, 0, 1] += float(y0)
    return c

def make_signature(gray_roi):
    """
    ROI signature: (merkez crop üzerinden)
    - resize
    - normalize
    - flatten
    """
    small = cv2.resize(gray_roi, (SIG_W, SIG_H), interpolation=cv2.INTER_AREA)
    v = small.astype(np.float32).reshape(-1)
    v -= v.mean()
    v /= (v.std() + 1e-6)
    return v

def cosine_sim(a, b):
    na = np.linalg.norm(a) + 1e-8
    nb = np.linalg.norm(b) + 1e-8
    return float(np.dot(a, b) / (na * nb))

def validate_corners_homography(corners, pattern, max_rmse_px=HOMOGRAPHY_RMSE_MAX):
    cols, rows = pattern
    N = cols * rows
    if corners is None or len(corners) != N:
        return False, np.inf

    pts = corners.reshape(-1, 2).astype(np.float32)
    grid = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2).astype(np.float32)

    H, _ = cv2.findHomography(grid, pts, method=0)
    if H is None:
        return False, np.inf

    proj = cv2.perspectiveTransform(grid.reshape(-1, 1, 2), H).reshape(-1, 2)
    rmse = float(np.sqrt(np.mean(np.sum((proj - pts) ** 2, axis=1))))
    return (rmse <= max_rmse_px), rmse

def reject_if_near_border(corners, w, h, border=BORDER_REJECT_PX):
    pts = corners.reshape(-1, 2)
    if (pts[:, 0].min() < border or pts[:, 0].max() > (w - border) or
        pts[:, 1].min() < border or pts[:, 1].max() > (h - border)):
        return True
    return False

def find_corners_scaled(gray_roi):
    """
    Corner detection on ROI:
    - preprocess
    - downscale
    - SB -> fallback classic
    - upscale back
    - subpix refine on preprocessed ROI
    """
    g = preprocess(gray_roi)

    if SCALE != 1.0:
        small = cv2.resize(g, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_AREA)
    else:
        small = g

    ok, corners = cv2.findChessboardCornersSB(
        small, PATTERN, flags=cv2.CALIB_CB_NORMALIZE_IMAGE
    )

    if (not ok) or corners is None:
        ok, corners = cv2.findChessboardCorners(
            small, PATTERN,
            flags=cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE
        )

    if (not ok) or corners is None:
        return None

    corners = corners.astype(np.float32)

    if SCALE != 1.0:
        corners /= float(SCALE)

    # subpix refine on ROI (preprocessed)
    cv2.cornerSubPix(g, corners, (11, 11), (-1, -1), subpix_crit)

    # border reject on ROI dims
    h, w = gray_roi.shape[:2]
    if reject_if_near_border(corners, w, h, border=BORDER_REJECT_PX):
        return None

    okH, rmse = validate_corners_homography(corners, PATTERN, HOMOGRAPHY_RMSE_MAX)
    if not okH:
        return None

    return corners

def collect_detections(video_path, label=""):
    """
    Returns list of dict:
      {
        "frame": i,
        "corners": corners_fullres,   # full-frame coords
        "img_size": (w,h),
        "centroid": (2,),
        "sig": signature_from_roi,
        "rmse": homography_rmse
      }
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Video açılamadı: {video_path}")

    det = []
    i = 0
    last_kept = -10**9
    img_size = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if (i % STRIDE_DET) == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            if img_size is None:
                img_size = (gray.shape[1], gray.shape[0])

            # center ROI
            roi, crop_info = center_crop(gray, CROP_FRAC)

            corners_roi = find_corners_scaled(roi)
            if corners_roi is not None:
                corners_full = uncrop_corners(corners_roi, crop_info)

                # full-frame border reject (extra safety)
                w, h = img_size
                if reject_if_near_border(corners_full, w, h, border=BORDER_REJECT_PX):
                    i += 1
                    continue

                if i - last_kept >= MIN_SEP_DET:
                    cen = corners_full.reshape(-1, 2).mean(axis=0)
                    sig = make_signature(preprocess(roi))
                    # rmse check computed on ROI corners (recompute quickly)
                    okH, rmse = validate_corners_homography(corners_roi, PATTERN, HOMOGRAPHY_RMSE_MAX)

                    det.append({
                        "frame": i,
                        "corners": corners_full,
                        "img_size": img_size,
                        "centroid": cen,
                        "sig": sig,
                        "rmse": float(rmse)
                    })
                    last_kept = i

                if len(det) >= MAX_DET:
                    break

        i += 1

    cap.release()
    print(f"{label} detections: {len(det)}")
    return det

def _fill_nan_xy(X):
    X = X.copy()
    for j in range(2):
        x = X[:, j]
        if np.isnan(x).all():
            return None
        for i in range(1, len(x)):
            if np.isnan(x[i]) and not np.isnan(x[i-1]):
                x[i] = x[i-1]
        for i in range(len(x)-2, -1, -1):
            if np.isnan(x[i]) and not np.isnan(x[i+1]):
                x[i] = x[i+1]
        X[:, j] = x
    return X

def estimate_lag_frames(detL, detR):
    if len(detL) < 5 or len(detR) < 5:
        return 0

    fL = np.array([d["frame"] for d in detL], dtype=np.int32)
    fR = np.array([d["frame"] for d in detR], dtype=np.int32)
    cL = np.array([d["centroid"] for d in detL], dtype=np.float64)
    cR = np.array([d["centroid"] for d in detR], dtype=np.float64)

    fmin = int(min(fL.min(), fR.min()))
    fmax = int(max(fL.max(), fR.max()))
    n = fmax - fmin + 1
    if n < 30:
        return int(np.median(fR) - np.median(fL))

    Xl = np.full((n, 2), np.nan, dtype=np.float64)
    Xr = np.full((n, 2), np.nan, dtype=np.float64)
    Xl[fL - fmin] = cL
    Xr[fR - fmin] = cR

    Xl = _fill_nan_xy(Xl)
    Xr = _fill_nan_xy(Xr)
    if Xl is None or Xr is None:
        return int(np.median(fR) - np.median(fL))

    sigL = np.linalg.norm(np.diff(Xl, axis=0), axis=1)
    sigR = np.linalg.norm(np.diff(Xr, axis=0), axis=1)
    sigL = (sigL - sigL.mean()) / (sigL.std() + 1e-9)
    sigR = (sigR - sigR.mean()) / (sigR.std() + 1e-9)

    corr = np.correlate(sigR, sigL, mode="full")
    lag = int(np.argmax(corr) - (len(sigL) - 1))
    lag = int(np.clip(lag, -MAX_LAG_FRAMES, MAX_LAG_FRAMES))
    return lag

def build_pairs_by_similarity(detL, detR, lag_frames, max_pairs):
    """
    For each left detection fL:
      target = fL + lag
      candidates R in [target-SEARCH_WIN, target+SEARCH_WIN]
      choose best cosine similarity on ROI signatures.
    1-to-1 R usage (no reuse).
    """
    r_by_frame = {d["frame"]: d for d in detR}
    r_frames = np.array(sorted(r_by_frame.keys()), dtype=np.int32)

    usedR = set()
    pairs = []
    img_size = detL[0]["img_size"]

    for dL in detL:
        fL = int(dL["frame"])
        target = fL + int(lag_frames)

        j0 = int(np.searchsorted(r_frames, target - SEARCH_WIN))
        j1 = int(np.searchsorted(r_frames, target + SEARCH_WIN, side="right"))

        best = None
        best_score = -1e9

        for j in range(j0, j1):
            if j < 0 or j >= len(r_frames):
                continue
            fR = int(r_frames[j])
            if fR in usedR:
                continue
            dR = r_by_frame[fR]
            if dR["img_size"] != img_size:
                continue

            score = cosine_sim(dL["sig"], dR["sig"])
            if score > best_score:
                best_score = score
                best = dR

        if best is None:
            continue

        usedR.add(int(best["frame"]))
        pairs.append((dL["corners"], best["corners"], img_size,
                      fL, int(best["frame"]), float(best_score),
                      float(dL["rmse"]), float(best["rmse"])))

        if len(pairs) >= max_pairs:
            break

    return pairs

def build_points(pairs):
    objpoints, imgL, imgR = [], [], []
    for cL, cR, *_ in pairs:
        objpoints.append(objp.copy())
        imgL.append(cL)
        imgR.append(cR)
    return objpoints, imgL, imgR

def stereo_rms_only(pairs):
    if len(pairs) < 12:
        return np.inf
    objpoints, imgL, imgR = build_points(pairs)
    img_size = pairs[0][2]

    _, K1, d1, *_ = cv2.calibrateCamera(objpoints, imgL, img_size, None, None,
                                        flags=INTRINSIC_FLAGS, criteria=criteria)
    _, K2, d2, *_ = cv2.calibrateCamera(objpoints, imgR, img_size, None, None,
                                        flags=INTRINSIC_FLAGS, criteria=criteria)

    retS, *_ = cv2.stereoCalibrate(objpoints, imgL, imgR,
                                   K1, d1, K2, d2, img_size,
                                   flags=stereo_flags_sweep, criteria=criteria)
    return float(retS)

def stereo_final(pairs):
    if len(pairs) < 15:
        raise RuntimeError(f"Final stereo için pair az: {len(pairs)} (>=15 önerilir).")
    objpoints, imgL, imgR = build_points(pairs)
    img_size = pairs[0][2]

    retL, K1, d1, *_ = cv2.calibrateCamera(objpoints, imgL, img_size, None, None,
                                           flags=INTRINSIC_FLAGS, criteria=criteria)
    retR, K2, d2, *_ = cv2.calibrateCamera(objpoints, imgR, img_size, None, None,
                                           flags=INTRINSIC_FLAGS, criteria=criteria)

    retS, K1, d1, K2, d2, R, T, E, F = cv2.stereoCalibrate(
        objpoints, imgL, imgR,
        K1, d1, K2, d2, img_size,
        flags=stereo_flags_final, criteria=criteria
    )
    return float(retL), float(retR), float(retS), K1, d1, K2, d2, R, T, E, F, img_size

def epipolar_errors(pairs, F):
    errs = []
    for cL, cR, *_ in pairs:
        ptsL = cL.reshape(-1, 2)
        ptsR = cR.reshape(-1, 2)

        ones = np.ones((ptsL.shape[0], 1), dtype=np.float64)
        x = np.hstack([ptsL.astype(np.float64), ones])  # Nx3

        l = (F @ x.T).T
        a, b, c = l[:, 0], l[:, 1], l[:, 2]

        x2 = ptsR[:, 0].astype(np.float64)
        y2 = ptsR[:, 1].astype(np.float64)

        dist = np.abs(a * x2 + b * y2 + c) / (np.sqrt(a*a + b*b) + 1e-12)
        errs.append(float(dist.mean()))
    return np.array(errs, dtype=np.float64)

def export_debug_pairs(pairs, left_path, right_path, out_dir, n_export=EXPORT_N):
    """
    İlk n_export pair'i yan yana JPG olarak kaydeder + CSV raporu.
    """
    capL = cv2.VideoCapture(left_path)
    capR = cv2.VideoCapture(right_path)

    def get_frame(cap, idx):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, fr = cap.read()
        return fr if ret else None

    rows = []
    for k, p in enumerate(pairs[:n_export]):
        cL, cR, _, fL, fR, sim, rmL, rmR = p

        frL = get_frame(capL, fL)
        frR = get_frame(capR, fR)
        if frL is None or frR is None:
            continue

        cv2.putText(frL, f"L {fL}  sim={sim:.3f} rmse={rmL:.2f}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(frR, f"R {fR}  sim={sim:.3f} rmse={rmR:.2f}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.drawChessboardCorners(frL, PATTERN, cL, True)
        cv2.drawChessboardCorners(frR, PATTERN, cR, True)

        h = max(frL.shape[0], frR.shape[0])
        w = frL.shape[1] + frR.shape[1]
        canvas = np.zeros((h, w, 3), dtype=np.uint8)
        canvas[:frL.shape[0], :frL.shape[1]] = frL
        canvas[:frR.shape[0], frL.shape[1]:frL.shape[1] + frR.shape[1]] = frR

        out_path = os.path.join(out_dir, "pairs", f"pair_L{fL}_R{fR}.jpg")
        cv2.imwrite(out_path, canvas)
        rows.append([k, fL, fR, f"{sim:.6f}", f"{rmL:.3f}", f"{rmR:.3f}", out_path])

    capL.release()
    capR.release()

    csv_path = os.path.join(out_dir, "pairs.csv")
    with open(csv_path, "w", newline="") as f:
        wcsv = csv.writer(f)
        wcsv.writerow(["idx", "frameL", "frameR", "cos_sim", "rmseL", "rmseR", "file"])
        wcsv.writerows(rows)

    print(f"[DEBUG] Exported {len(rows)} pairs -> {os.path.abspath(os.path.join(out_dir, 'pairs'))}")
    print(f"[DEBUG] CSV -> {os.path.abspath(csv_path)}")


# ============================================================
# MAIN
# ============================================================
print("Collecting detections...")
detL = collect_detections(LEFT_VIDEO, label="LEFT")
detR = collect_detections(RIGHT_VIDEO, label="RIGHT")

print(f"Detections: Left={len(detL)}, Right={len(detR)}")
print("Left frames (first 10):", [d["frame"] for d in detL[:10]])
print("Right frames (first 10):", [d["frame"] for d in detR[:10]])

if len(detL) < 12 or len(detR) < 12:
    raise RuntimeError("Detections çok az. STRIDE_DET=1 yap, CROP_FRAC=0.8 dene, MAX_DET artır.")

lag0 = estimate_lag_frames(detL, detR)
print("Initial lag estimate (frames):", lag0)

# ---- LAG SWEEP ----
best_rms = 1e18
best_lag = None

for lag_try in range(lag0 - LAG_SWEEP_HALF, lag0 + LAG_SWEEP_HALF + 1):
    pairs_try = build_pairs_by_similarity(detL, detR, lag_try, MAX_PAIRS_SWEEP)
    if len(pairs_try) < 12:
        continue
    rms = stereo_rms_only(pairs_try)
    if rms < best_rms:
        best_rms = rms
        best_lag = lag_try

if best_lag is None:
    raise RuntimeError("Lag sweep başarısız. SEARCH_WIN=4 yap veya LAG_SWEEP_HALF=80 dene.")

print(f"Best lag: {best_lag} | Sweep RMS (~{MAX_PAIRS_SWEEP} pairs): {best_rms:.4f}")

# ---- FINAL PAIRS ----
pairs_final = build_pairs_by_similarity(detL, detR, best_lag, MAX_PAIRS_FINAL)
print("Pairs for final:", len(pairs_final))

if len(pairs_final) < 15:
    raise RuntimeError("Final pair az. STRIDE_DET=1 yap veya MAX_DET artır veya SEARCH_WIN=4 dene.")

# (İstersen) debug export: eşleşmeleri gözle kontrol
if EXPORT_PAIRS_JPG:
    export_debug_pairs(pairs_final, LEFT_VIDEO, RIGHT_VIDEO, DEBUG_DIR, n_export=EXPORT_N)

# ---- FINAL CALIB (pre-clean) ----
retL, retR, retS, K1, d1, K2, d2, R, T, E, F, img_size = stereo_final(pairs_final)
print("=== PRE-CLEAN RESULTS ===")
print("Image size:", img_size)
print("RMS L/R/S:", retL, retR, retS)
print("T (mm):", T.ravel(), "|T|:", float(np.linalg.norm(T.ravel())))

# ---- EPIPOLAR OUTLIER CLEAN ----
errs = epipolar_errors(pairs_final, F)
thr = np.quantile(errs, OUTLIER_DROP_Q)
pairs_clean = [p for p, e in zip(pairs_final, errs) if e <= thr]
print(f"Epipolar cleaning: keeping {len(pairs_clean)}/{len(pairs_final)} (thr={thr:.3f} px)")

# ---- FINAL CALIB (clean) ----
retL2, retR2, retS2, K1c, d1c, K2c, d2c, Rc, Tc, Ec, Fc, _ = stereo_final(pairs_clean)
print("=== FINAL CLEAN RESULTS ===")
print("RMS L/R/S:", retL2, retR2, retS2)
print("T (mm):", Tc.ravel(), "|T|:", float(np.linalg.norm(Tc.ravel())))

# ---- SAVE ----
np.savez(
    "stereo_calib_final.npz",
    K1=K1c, d1=d1c, K2=K2c, d2=d2c,
    R=Rc, T=Tc, E=Ec, F=Fc,
    img_size=np.array(img_size),
    best_lag=np.array([best_lag], dtype=np.int32),
    sweep_rms=np.array([best_rms], dtype=np.float64),
    stride_det=np.array([STRIDE_DET], dtype=np.int32),
    scale=np.array([SCALE], dtype=np.float64),
    crop_frac=np.array([CROP_FRAC], dtype=np.float64),
    search_win=np.array([SEARCH_WIN], dtype=np.int32),
    used_pairs=np.array([len(pairs_clean)], dtype=np.int32),
    square_size_mm=np.array([SQUARE_SIZE_MM], dtype=np.float64),
    epipolar_errs=errs,
)
print("Saved: stereo_calib_final.npz")
print(f"Debug folder: {os.path.abspath(DEBUG_DIR)}")


LEFT detections: 95
RIGHT detections: 306
Detections: Left=95, Right=306
Left frames (first 10): [86, 88, 90, 92, 94, 96, 98, 100, 102, 104]
Right frames (first 10): [94, 98, 100, 102, 104, 106, 108, 110, 112, 114]
Initial lag estimate (frames): -8
Best lag: -21 | Sweep RMS (~20 pairs): 0.4418
Pairs for final: 73
[DEBUG] Exported 73 pairs -> C:\Users\edanu\OneDrive - Koc Universitesi\Masaüstü\Koc_Calismalar\AUTOTRACKING_dennis\Autotracking Version 2\stereo_debug\pairs
[DEBUG] CSV -> C:\Users\edanu\OneDrive - Koc Universitesi\Masaüstü\Koc_Calismalar\AUTOTRACKING_dennis\Autotracking Version 2\stereo_debug\pairs.csv
=== PRE-CLEAN RESULTS ===
Image size: (1280, 720)
RMS L/R/S: 0.3162633433773764 0.3519469395993643 27.329569367870164
T (mm): [1212.38516205 4112.93495425 3757.32565735] |T|: 5701.19354297349
Epipolar cleaning: keeping 51/73 (thr=7725.783 px)
=== FINAL CLEAN RESULTS ===
RMS L/R/S: 0.32166740636318986 0.365291278471161 11.930189460729645
T (mm): [2804.13638688  657.55974097 868

In [1]:
import os
import cv2 as cv
import numpy as np

# -------------------------
# USER SETTINGS
# -------------------------
left_video  = "calib8_left.mp4"
right_video = "calib8_right.mp4"

chessboardSize = (6, 5)      # (cols, rows) internal corners
square_size_mm = 14.0
frameSize = (1920, 1080)     # (w,h)


RIGHT_MODE = "id"            


MAX_SAMPLES = 120            
SKIP = 1                     
START_FRAME = 0


criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)
subpix_win = (11, 11)


DEBUG_EVERY = 20

def apply_mode(img, mode):
    if mode == "id": return img
    if mode == "flip_x": return cv.flip(img, 1)
    if mode == "flip_y": return cv.flip(img, 0)
    if mode == "flip_xy": return cv.flip(img, -1)
    raise ValueError(mode)

def build_objp(board_size, sq_mm):
    cols, rows = board_size
    objp = np.zeros((rows*cols, 3), np.float32)
    objp[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2)
    objp *= float(sq_mm)
    return objp

def reorder_corners_for_mode(corners, pattern, mode):
    cols, rows = pattern
    c = corners.reshape(rows, cols, 2).copy()
    if mode == "id":
        pass
    elif mode == "flip_x":
        c = c[:, ::-1, :]
    elif mode == "flip_y":
        c = c[::-1, :, :]
    elif mode == "flip_xy":
        c = c[::-1, ::-1, :]
    else:
        raise ValueError(mode)
    return c.reshape(-1, 1, 2).astype(np.float32)

objp = build_objp(chessboardSize, square_size_mm)

objpoints = []
imgpointsL = []
imgpointsR = []

capL = cv.VideoCapture(left_video)
capR = cv.VideoCapture(right_video)
if not capL.isOpened() or not capR.isOpened():
    raise RuntimeError("Videolar açılamadı. Yol kontrol et.")

capL.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)
capR.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)

accepted = 0
i = START_FRAME

while True:
    retL, frameL = capL.read()
    retR, frameR0 = capR.read()
    if not retL or not retR:
        break

    if (i - START_FRAME) % SKIP != 0:
        i += 1
        continue

    frameR = apply_mode(frameR0, RIGHT_MODE)

    grayL = cv.cvtColor(frameL, cv.COLOR_BGR2GRAY)
    grayR = cv.cvtColor(frameR, cv.COLOR_BGR2GRAY)

    retCL, cornersL = cv.findChessboardCorners(grayL, chessboardSize, None)
    retCR, cornersR = cv.findChessboardCorners(grayR, chessboardSize, None)

    if retCL and retCR:
        cornersL = cv.cornerSubPix(grayL, cornersL, subpix_win, (-1,-1), criteria)
        cornersR = cv.cornerSubPix(grayR, cornersR, subpix_win, (-1,-1), criteria)

        
        cornersR = reorder_corners_for_mode(cornersR, chessboardSize, RIGHT_MODE)

        objpoints.append(objp)
        imgpointsL.append(cornersL)
        imgpointsR.append(cornersR)

        accepted += 1

        if accepted % DEBUG_EVERY == 0:
            visL = frameL.copy()
            visR = frameR.copy()
            cv.drawChessboardCorners(visL, chessboardSize, cornersL, True)
            cv.drawChessboardCorners(visR, chessboardSize, cornersR, True)
            cv.imwrite(f"dbg_L_{accepted:03d}.png", visL)
            cv.imwrite(f"dbg_R_{accepted:03d}.png", visR)
            print(f"[collect] accepted={accepted}")

        if accepted >= MAX_SAMPLES:
            break

    i += 1

capL.release(); capR.release()

print(f"[done] collected pairs: {accepted}")
if accepted < 15:
    raise RuntimeError("Çok az pair toplandı. chessboardSize / görüntü kalitesi / RIGHT_MODE kontrol et.")


retL, K1, dist1, rvecsL, tvecsL = cv.calibrateCamera(objpoints, imgpointsL, frameSize, None, None)
retR, K2, dist2, rvecsR, tvecsR = cv.calibrateCamera(objpoints, imgpointsR, frameSize, None, None)

print(f"[single] RMS L: {retL:.4f} px")
print(f"[single] RMS R: {retR:.4f} px")


flags = 0
flags |= cv.CALIB_FIX_INTRINSIC
criteria_stereo = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)

retStereo, K1s, dist1s, K2s, dist2s, R, T, E, F = cv.stereoCalibrate(
    objpoints, imgpointsL, imgpointsR,
    K1, dist1, K2, dist2, frameSize,
    criteria=criteria_stereo, flags=flags
)

print(f"[stereo] RMS: {retStereo:.4f} px")
print(f"[stereo] baseline |T|: {float(np.linalg.norm(T)):.3f} mm")


rectifyScale = 0.2
R1, R2, P1, P2, Q, roi1, roi2 = cv.stereoRectify(
    K1, dist1, K2, dist2, frameSize, R, T, alpha=rectifyScale
)

map1L, map2L = cv.initUndistortRectifyMap(K1, dist1, R1, P1, frameSize, cv.CV_16SC2)
map1R, map2R = cv.initUndistortRectifyMap(K2, dist2, R2, P2, frameSize, cv.CV_16SC2)


capL = cv.VideoCapture(left_video); capR = cv.VideoCapture(right_video)
capL.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)
capR.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)
_, fL = capL.read()
_, fR0 = capR.read()
capL.release(); capR.release()
fR = apply_mode(fR0, RIGHT_MODE)

rectL = cv.remap(fL, map1L, map2L, cv.INTER_LINEAR)
rectR = cv.remap(fR, map1R, map2R, cv.INTER_LINEAR)

sbs = np.hstack([rectL, rectR])
step = max(40, frameSize[1] // 20)
for y in range(0, sbs.shape[0], step):
    cv.line(sbs, (0, y), (sbs.shape[1]-1, y), (0, 255, 0), 1)
cv.imwrite("rectified_side_by_side.png", sbs)
print("[save] rectified_side_by_side.png")

fs = cv.FileStorage("stereoMap.xml", cv.FILE_STORAGE_WRITE)
fs.write("stereoMapL_x", map1L)
fs.write("stereoMapL_y", map2L)
fs.write("stereoMapR_x", map1R)
fs.write("stereoMapR_y", map2R)
fs.release()
print("[save] stereoMap.xml")


np.savez(
    "stereo_calib_video.npz",
    K1=K1, dist1=dist1, K2=K2, dist2=dist2,
    R=R, T=T, E=E, F=F,
    R1=R1, R2=R2, P1=P1, P2=P2, Q=Q,
    frameSize=np.array([frameSize[0], frameSize[1]], dtype=np.int32),
    chessboardSize=np.array([chessboardSize[0], chessboardSize[1]], dtype=np.int32),
    square_size_mm=float(square_size_mm),
    RIGHT_MODE=RIGHT_MODE,
    rms_single_L=float(retL),
    rms_single_R=float(retR),
    rms_stereo=float(retStereo),
    n_pairs=int(accepted),
)
print("[save] stereo_calib_video.npz")

[collect] accepted=20
[collect] accepted=40
[collect] accepted=60
[collect] accepted=80
[collect] accepted=100
[collect] accepted=120
[done] collected pairs: 120
[single] RMS L: 0.3265 px
[single] RMS R: 0.9139 px
[stereo] RMS: 1.9589 px
[stereo] baseline |T|: 273.260 mm
[save] rectified_side_by_side.png
[save] stereoMap.xml
[save] stereo_calib_video.npz


In [2]:
def split_distortion(dist):
    """
    OpenCV distortion -> radial & tangential
    Supports pinhole + rational model
    """
    d = np.asarray(dist, dtype=float).ravel()

    # Tangential
    tangential = d[2:4] if d.size >= 4 else np.zeros(2)

    # Radial (k1 k2 k3 k4 k5 k6)
    radial = []
    if d.size >= 1: radial.append(d[0])  # k1
    if d.size >= 2: radial.append(d[1])  # k2
    if d.size >= 5: radial.append(d[4])  # k3
    if d.size >= 6: radial.append(d[5])  # k4
    if d.size >= 7: radial.append(d[6])  # k5
    if d.size >= 8: radial.append(d[7])  # k6

    return np.array(radial, dtype=float), np.array(tangential, dtype=float)

from scipy.io import savemat
import numpy as np

# --- Distortion ayrıştır ---
radial1, tang1 = split_distortion(dist1)
radial2, tang2 = split_distortion(dist2)

# --- MATLAB dictionary ---
mat_dict = {
    # Intrinsics
    'K1': K1.astype(float),
    'K2': K2.astype(float),

    # Full distortion (OpenCV order – debug için faydalı)
    'dist1_full': np.asarray(dist1, dtype=float),
    'dist2_full': np.asarray(dist2, dtype=float),

    # Separated distortion (MATLAB-friendly)
    'radial1': radial1,
    'tangential1': tang1,
    'radial2': radial2,
    'tangential2': tang2,

    # Stereo geometry
    'R': R.astype(float),
    'T': T.astype(float),
    'E': np.asarray(E, dtype=float),
    'F': np.asarray(F, dtype=float),

    # Rectification
    'R1': R1.astype(float),
    'R2': R2.astype(float),
    'P1': P1.astype(float),
    'P2': P2.astype(float),
    'Q':  Q.astype(float),

    # Meta
    'imageSize': np.array([frameSize[1], frameSize[0]], dtype=np.int32),  # [H W]
    'boardSize_rc': np.array([chessboardSize[1], chessboardSize[0]], dtype=np.int32),
    'squareSizeMM': float(square_size_mm),

    # Errors
    'rms_single_L': float(retL),
    'rms_single_R': float(retR),
    'rms_stereo':   float(retStereo),

    # Model info
    'model': 'opencv_pinhole_rational',
}

savemat("stereo_from_python_koc.mat", mat_dict)
print("[save] Kaydedildi: stereo_from_python_koc.mat")


[save] Kaydedildi: stereo_from_python_koc.mat


In [1]:
import os
import cv2 as cv
import numpy as np

# ============================================================
# STEREO CALIBRATION FROM ONE "STACKED" VIDEO
#   stacked video layout:
#     TOP  = LEFT camera
#     BOTTOM = RIGHT camera
#
# Workflow:
#   1) read stacked frame
#   2) split into (frameL, frameR)
#   3) detect chessboard in both
#   4) collect pairs
#   5) calibrate intrinsics + stereo
#   6) rectify + save maps + debug images + npz
# ============================================================

# -------------------------
# USER SETTINGS
# -------------------------
STACKED_VIDEO = "Syncronized.mp4"   # Premiere export (top=left, bottom=right)

OUT_DIR = "stereo_from_stacked_debug"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "dbg_pairs"), exist_ok=True)

CHESSBOARD = (6, 5)       # (cols, rows) inner corners
SQUARE_SIZE_MM = 130.0     # set your real square size (mm)

# Sampling
START_FRAME = 0
SKIP = 1                  # 1=every frame, 2=every 2 frames...
MAX_SAMPLES = 120         # how many accepted pairs to collect
DEBUG_EVERY = 20          # save overlay debug every N accepted

# Right frame orientation (rarely needed if stacked is correct)
RIGHT_MODE = "id"         # "id", "flip_x", "flip_y", "flip_xy"

# Detection / refinement
SUBPIX_WIN = (11, 11)
criteria_subpix = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 1e-3)
criteria_stereo = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 60, 1e-6)

# Calibration flags
# For wide-angle GoPro, rational model often helps:
INTRINSIC_FLAGS = cv.CALIB_RATIONAL_MODEL
STEREO_FLAGS = cv.CALIB_FIX_INTRINSIC  # keep intrinsics fixed in stereo step

# Rectify
RECTIFY_ALPHA = 0.2       # 0=crop more, 1=keep more black borders


# -------------------------
# HELPERS
# -------------------------
def apply_mode(img, mode):
    if mode == "id":
        return img
    if mode == "flip_x":
        return cv.flip(img, 1)
    if mode == "flip_y":
        return cv.flip(img, 0)
    if mode == "flip_xy":
        return cv.flip(img, -1)
    raise ValueError(f"Unknown mode: {mode}")

def build_objp(board_size, sq_mm):
    cols, rows = board_size
    objp = np.zeros((rows * cols, 3), np.float32)
    objp[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2)
    objp *= float(sq_mm)
    return objp

def reorder_corners_for_mode(corners, pattern, mode):
    """
    If you flip the RIGHT image, the corner ordering must be flipped too.
    """
    cols, rows = pattern
    c = corners.reshape(rows, cols, 2).copy()
    if mode == "id":
        pass
    elif mode == "flip_x":
        c = c[:, ::-1, :]
    elif mode == "flip_y":
        c = c[::-1, :, :]
    elif mode == "flip_xy":
        c = c[::-1, ::-1, :]
    else:
        raise ValueError(mode)
    return c.reshape(-1, 1, 2).astype(np.float32)

def split_stacked(frame):
    """
    Split stacked frame into top/bottom halves.
    Returns (frameL, frameR).
    """
    h, w = frame.shape[:2]
    if h % 2 != 0:
        # if odd height due to export, crop last row
        frame = frame[:h-1, :, :]
        h -= 1
    half = h // 2
    top = frame[:half, :, :]
    bot = frame[half:, :, :]
    return top, bot

def find_corners(gray, board):
    """
    Robust chessboard detection:
      - try SB first (more robust)
      - fallback to classic
    """
    ok, corners = cv.findChessboardCornersSB(gray, board, flags=cv.CALIB_CB_NORMALIZE_IMAGE)
    if not ok or corners is None:
        ok, corners = cv.findChessboardCorners(
            gray, board,
            flags=cv.CALIB_CB_ADAPTIVE_THRESH | cv.CALIB_CB_NORMALIZE_IMAGE
        )
    if not ok or corners is None:
        return None
    corners = cv.cornerSubPix(gray, corners, SUBPIX_WIN, (-1, -1), criteria_subpix)
    return corners

def draw_and_save_pair(visL, visR, cornersL, cornersR, idx, frame_idx):
    """
    Save overlay debug images and a side-by-side for quick inspection.
    """
    a = visL.copy()
    b = visR.copy()
    cv.drawChessboardCorners(a, CHESSBOARD, cornersL, True)
    cv.drawChessboardCorners(b, CHESSBOARD, cornersR, True)

    cv.putText(a, f"L frame={frame_idx}", (20, 40), cv.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
    cv.putText(b, f"R frame={frame_idx}", (20, 40), cv.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    sbs = np.hstack([a, b])
    out = os.path.join(OUT_DIR, "dbg_pairs", f"pair_{idx:03d}_f{frame_idx}.png")
    cv.imwrite(out, sbs)
    return out


# -------------------------
# MAIN
# -------------------------
objp = build_objp(CHESSBOARD, SQUARE_SIZE_MM)

objpoints = []
imgpointsL = []
imgpointsR = []

cap = cv.VideoCapture(STACKED_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Video açılamadı: {STACKED_VIDEO}")

# get actual size from first frame (more reliable than hardcoding)
cap.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)
ret, first = cap.read()
if not ret:
    raise RuntimeError("İlk frame okunamadı.")
frameL0, frameR0 = split_stacked(first)
frameR0 = apply_mode(frameR0, RIGHT_MODE)

hL, wL = frameL0.shape[:2]
hR, wR = frameR0.shape[:2]
if (hL, wL) != (hR, wR):
    raise RuntimeError("Split sonrası L/R boyutları farklı. Premiere export ayarlarını kontrol et.")
frameSize = (wL, hL)  # (w,h)

print(f"[info] frameSize from stacked video halves: {frameSize}")

# rewind to start
cap.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)

accepted = 0
frame_idx = START_FRAME
saved_debug = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if (frame_idx - START_FRAME) % SKIP != 0:
        frame_idx += 1
        continue

    frameL, frameR0 = split_stacked(frame)
    frameR = apply_mode(frameR0, RIGHT_MODE)

    grayL = cv.cvtColor(frameL, cv.COLOR_BGR2GRAY)
    grayR = cv.cvtColor(frameR, cv.COLOR_BGR2GRAY)

    cornersL = find_corners(grayL, CHESSBOARD)
    cornersR = find_corners(grayR, CHESSBOARD)

    if cornersL is not None and cornersR is not None:
        # if RIGHT was flipped, reorder corner indexing accordingly
        cornersR = reorder_corners_for_mode(cornersR, CHESSBOARD, RIGHT_MODE)

        objpoints.append(objp.copy())
        imgpointsL.append(cornersL)
        imgpointsR.append(cornersR)

        accepted += 1

        if accepted % DEBUG_EVERY == 0:
            out = draw_and_save_pair(frameL, frameR, cornersL, cornersR, accepted, frame_idx)
            saved_debug.append(out)
            print(f"[collect] accepted={accepted} at stacked_frame={frame_idx}  | dbg={out}")

        if accepted >= MAX_SAMPLES:
            break

    frame_idx += 1

cap.release()

print(f"[done] collected pairs: {accepted}")
if accepted < 15:
    raise RuntimeError("Çok az pair toplandı. Chessboard görünürlüğü / ışık / motion blur kontrol et.")

# ---------- Intrinsics ----------
rmsL, K1, dist1, rvecsL, tvecsL = cv.calibrateCamera(
    objpoints, imgpointsL, frameSize, None, None, flags=INTRINSIC_FLAGS, criteria=criteria_stereo
)
rmsR, K2, dist2, rvecsR, tvecsR = cv.calibrateCamera(
    objpoints, imgpointsR, frameSize, None, None, flags=INTRINSIC_FLAGS, criteria=criteria_stereo
)

print(f"[single] RMS L: {rmsL:.4f} px")
print(f"[single] RMS R: {rmsR:.4f} px")

# ---------- Stereo ----------
retStereo, K1s, dist1s, K2s, dist2s, R, T, E, F = cv.stereoCalibrate(
    objpoints, imgpointsL, imgpointsR,
    K1, dist1, K2, dist2, frameSize,
    criteria=criteria_stereo, flags=STEREO_FLAGS
)

print(f"[stereo] RMS: {retStereo:.4f} px")
print(f"[stereo] baseline |T|: {float(np.linalg.norm(T)):.3f} mm")
print(f"[stereo] T (mm): {T.ravel()}")

# ---------- Rectification ----------
R1, R2, P1, P2, Q, roi1, roi2 = cv.stereoRectify(
    K1, dist1, K2, dist2, frameSize, R, T, alpha=RECTIFY_ALPHA
)

map1L, map2L = cv.initUndistortRectifyMap(K1, dist1, R1, P1, frameSize, cv.CV_16SC2)
map1R, map2R = cv.initUndistortRectifyMap(K2, dist2, R2, P2, frameSize, cv.CV_16SC2)

# ---------- Quick rectified preview (from a sample frame) ----------
cap = cv.VideoCapture(STACKED_VIDEO)
cap.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)
ret, frame = cap.read()
cap.release()

if ret:
    fL, fR0 = split_stacked(frame)
    fR = apply_mode(fR0, RIGHT_MODE)

    rectL = cv.remap(fL, map1L, map2L, cv.INTER_LINEAR)
    rectR = cv.remap(fR, map1R, map2R, cv.INTER_LINEAR)

    sbs = np.hstack([rectL, rectR])
    step = max(40, frameSize[1] // 20)
    for y in range(0, sbs.shape[0], step):
        cv.line(sbs, (0, y), (sbs.shape[1]-1, y), (0, 255, 0), 1)

    out_rect = os.path.join(OUT_DIR, "rectified_side_by_side.png")
    cv.imwrite(out_rect, sbs)
    print(f"[save] {out_rect}")

# ---------- Save xml maps ----------
xml_path = os.path.join(OUT_DIR, "stereoMap.xml")
fs = cv.FileStorage(xml_path, cv.FILE_STORAGE_WRITE)
fs.write("stereoMapL_x", map1L)
fs.write("stereoMapL_y", map2L)
fs.write("stereoMapR_x", map1R)
fs.write("stereoMapR_y", map2R)
fs.release()
print(f"[save] {xml_path}")

# ---------- Save NPZ ----------
npz_path = os.path.join(OUT_DIR, "stereo_calib_stacked.npz")
np.savez(
    npz_path,
    K1=K1, dist1=dist1, K2=K2, dist2=dist2,
    R=R, T=T, E=E, F=F,
    R1=R1, R2=R2, P1=P1, P2=P2, Q=Q,
    frameSize=np.array([frameSize[0], frameSize[1]], dtype=np.int32),
    chessboardSize=np.array([CHESSBOARD[0], CHESSBOARD[1]], dtype=np.int32),
    square_size_mm=float(SQUARE_SIZE_MM),
    RIGHT_MODE=RIGHT_MODE,
    rms_single_L=float(rmsL),
    rms_single_R=float(rmsR),
    rms_stereo=float(retStereo),
    n_pairs=int(accepted),
    debug_pairs=np.array(saved_debug, dtype=object)
)
print(f"[save] {npz_path}")


[info] frameSize from stacked video halves: (1920, 540)
[collect] accepted=20 at stacked_frame=19  | dbg=stereo_from_stacked_debug\dbg_pairs\pair_020_f19.png
[collect] accepted=40 at stacked_frame=39  | dbg=stereo_from_stacked_debug\dbg_pairs\pair_040_f39.png
[collect] accepted=60 at stacked_frame=59  | dbg=stereo_from_stacked_debug\dbg_pairs\pair_060_f59.png
[collect] accepted=80 at stacked_frame=79  | dbg=stereo_from_stacked_debug\dbg_pairs\pair_080_f79.png
[collect] accepted=100 at stacked_frame=99  | dbg=stereo_from_stacked_debug\dbg_pairs\pair_100_f99.png
[collect] accepted=120 at stacked_frame=119  | dbg=stereo_from_stacked_debug\dbg_pairs\pair_120_f119.png
[done] collected pairs: 120
[single] RMS L: 0.1768 px
[single] RMS R: 0.1922 px
[stereo] RMS: 7.8735 px
[stereo] baseline |T|: 2646.769 mm
[stereo] T (mm): [-2108.19148162  1599.71269433    42.84501978]
[save] stereo_from_stacked_debug\rectified_side_by_side.png
[save] stereo_from_stacked_debug\stereoMap.xml
[save] stereo_from

In [1]:
import os
import cv2 as cv
import numpy as np

# =========================
# USER SETTINGS
# =========================
STACKED_VIDEO = "Syncronized.mp4"     # top=left, bottom=right

OUT_DIR = "stereo_from_stacked_debug_v2"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "dbg_pairs"), exist_ok=True)

CHESSBOARD = (6, 5)        # (cols, rows)
SQUARE_SIZE_MM = 14.0

START_FRAME = 0
SKIP = 1
MAX_SAMPLES = 120
DEBUG_EVERY = 20

# If you flipped RIGHT half visually, set this
RIGHT_MODE = "id"          # "id", "flip_x", "flip_y", "flip_xy"

# IMPORTANT:
# If Premiere made each half 540p (1920x540), we upscale back to 1080p
UPSCALE_TO_H = 1080        # set None to disable upscaling

SUBPIX_WIN = (11, 11)
criteria_subpix = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 1e-3)
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 80, 1e-7)

INTRINSIC_FLAGS = cv.CALIB_RATIONAL_MODEL
STEREO_FLAGS = cv.CALIB_USE_INTRINSIC_GUESS   # allow slight adjustment

RECTIFY_ALPHA = 0.2


# =========================
# HELPERS
# =========================
def apply_mode(img, mode):
    if mode == "id": return img
    if mode == "flip_x": return cv.flip(img, 1)
    if mode == "flip_y": return cv.flip(img, 0)
    if mode == "flip_xy": return cv.flip(img, -1)
    raise ValueError(mode)

def build_objp(board_size, sq_mm):
    cols, rows = board_size
    objp = np.zeros((rows * cols, 3), np.float32)
    objp[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2)
    objp *= float(sq_mm)
    return objp

def reorder_corners_for_mode(corners, pattern, mode):
    cols, rows = pattern
    c = corners.reshape(rows, cols, 2).copy()
    if mode == "id":
        pass
    elif mode == "flip_x":
        c = c[:, ::-1, :]
    elif mode == "flip_y":
        c = c[::-1, :, :]
    elif mode == "flip_xy":
        c = c[::-1, ::-1, :]
    else:
        raise ValueError(mode)
    return c.reshape(-1, 1, 2).astype(np.float32)

def split_stacked(frame):
    h, w = frame.shape[:2]
    if h % 2 != 0:
        frame = frame[:h-1]
        h -= 1
    half = h // 2
    top = frame[:half].copy()
    bot = frame[half:].copy()
    return top, bot

def maybe_upscale(img, target_h):
    if target_h is None:
        return img
    h, w = img.shape[:2]
    if h == target_h:
        return img
    scale = target_h / float(h)
    new_w = int(round(w * scale))
    return cv.resize(img, (new_w, target_h), interpolation=cv.INTER_CUBIC)

def find_corners(gray, board):
    ok, corners = cv.findChessboardCornersSB(gray, board, flags=cv.CALIB_CB_NORMALIZE_IMAGE)
    if not ok or corners is None:
        ok, corners = cv.findChessboardCorners(
            gray, board,
            flags=cv.CALIB_CB_ADAPTIVE_THRESH | cv.CALIB_CB_NORMALIZE_IMAGE
        )
    if not ok or corners is None:
        return None
    corners = cv.cornerSubPix(gray, corners, SUBPIX_WIN, (-1, -1), criteria_subpix)
    return corners

def save_debug_pair(frameL, frameR, cornersL, cornersR, accepted, frame_idx):
    a = frameL.copy()
    b = frameR.copy()
    cv.drawChessboardCorners(a, CHESSBOARD, cornersL, True)
    cv.drawChessboardCorners(b, CHESSBOARD, cornersR, True)
    sbs = np.hstack([a, b])
    out = os.path.join(OUT_DIR, "dbg_pairs", f"pair_{accepted:03d}_f{frame_idx}.png")
    cv.imwrite(out, sbs)
    return out


# =========================
# RUN
# =========================
objp = build_objp(CHESSBOARD, SQUARE_SIZE_MM)

objpoints, imgpointsL, imgpointsR = [], [], []

cap = cv.VideoCapture(STACKED_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Video açılamadı: {STACKED_VIDEO}")

cap.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)
ret, first = cap.read()
if not ret:
    raise RuntimeError("İlk frame okunamadı.")

L0, R0 = split_stacked(first)
R0 = apply_mode(R0, RIGHT_MODE)

L0 = maybe_upscale(L0, UPSCALE_TO_H)
R0 = maybe_upscale(R0, UPSCALE_TO_H)

if L0.shape[:2] != R0.shape[:2]:
    raise RuntimeError("Up-scale sonrası L/R boyutu farklı çıktı. Premiere export sorunlu.")

frameSize = (L0.shape[1], L0.shape[0])  # (w,h)
print(f"[info] half frameSize after upscale: {frameSize}")

cap.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)

accepted = 0
frame_idx = START_FRAME

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if (frame_idx - START_FRAME) % SKIP != 0:
        frame_idx += 1
        continue

    frameL, frameR0 = split_stacked(frame)
    frameR = apply_mode(frameR0, RIGHT_MODE)

    frameL = maybe_upscale(frameL, UPSCALE_TO_H)
    frameR = maybe_upscale(frameR, UPSCALE_TO_H)

    grayL = cv.cvtColor(frameL, cv.COLOR_BGR2GRAY)
    grayR = cv.cvtColor(frameR, cv.COLOR_BGR2GRAY)

    cornersL = find_corners(grayL, CHESSBOARD)
    cornersR = find_corners(grayR, CHESSBOARD)

    if cornersL is not None and cornersR is not None:
        cornersR = reorder_corners_for_mode(cornersR, CHESSBOARD, RIGHT_MODE)

        objpoints.append(objp.copy())
        imgpointsL.append(cornersL)
        imgpointsR.append(cornersR)

        accepted += 1
        if accepted % DEBUG_EVERY == 0:
            out = save_debug_pair(frameL, frameR, cornersL, cornersR, accepted, frame_idx)
            print(f"[collect] accepted={accepted} at frame={frame_idx} | dbg={out}")

        if accepted >= MAX_SAMPLES:
            break

    frame_idx += 1

cap.release()

print(f"[done] collected pairs: {accepted}")
if accepted < 15:
    raise RuntimeError("Çok az pair. Görüntü netliği / chessboard boyutu / poz / ışık kontrol et.")

# --- Intrinsics ---
rmsL, K1, dist1, *_ = cv.calibrateCamera(objpoints, imgpointsL, frameSize, None, None,
                                         flags=INTRINSIC_FLAGS, criteria=criteria)
rmsR, K2, dist2, *_ = cv.calibrateCamera(objpoints, imgpointsR, frameSize, None, None,
                                         flags=INTRINSIC_FLAGS, criteria=criteria)

print(f"[single] RMS L: {rmsL:.4f} px")
print(f"[single] RMS R: {rmsR:.4f} px")

# --- Stereo (allow slight intrinsic adjustment) ---
retStereo, K1s, dist1s, K2s, dist2s, R, T, E, F = cv.stereoCalibrate(
    objpoints, imgpointsL, imgpointsR,
    K1, dist1, K2, dist2, frameSize,
    criteria=criteria,
    flags=STEREO_FLAGS
)

print(f"[stereo] RMS: {retStereo:.4f} px")
print(f"[stereo] baseline |T|: {float(np.linalg.norm(T)):.3f} mm")
print(f"[stereo] T (mm): {T.ravel()}")

# --- Rectify ---
R1, R2, P1, P2, Q, *_ = cv.stereoRectify(K1s, dist1s, K2s, dist2s, frameSize, R, T, alpha=RECTIFY_ALPHA)
map1L, map2L = cv.initUndistortRectifyMap(K1s, dist1s, R1, P1, frameSize, cv.CV_16SC2)
map1R, map2R = cv.initUndistortRectifyMap(K2s, dist2s, R2, P2, frameSize, cv.CV_16SC2)

# preview rectified
cap = cv.VideoCapture(STACKED_VIDEO)
cap.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)
ret, frame = cap.read()
cap.release()
if ret:
    fL, fR0 = split_stacked(frame)
    fR = apply_mode(fR0, RIGHT_MODE)
    fL = maybe_upscale(fL, UPSCALE_TO_H)
    fR = maybe_upscale(fR, UPSCALE_TO_H)

    rectL = cv.remap(fL, map1L, map2L, cv.INTER_LINEAR)
    rectR = cv.remap(fR, map1R, map2R, cv.INTER_LINEAR)

    sbs = np.hstack([rectL, rectR])
    step = max(30, frameSize[1] // 18)
    for y in range(0, sbs.shape[0], step):
        cv.line(sbs, (0, y), (sbs.shape[1]-1, y), (0,255,0), 1)

    out_rect = os.path.join(OUT_DIR, "rectified_side_by_side.png")
    cv.imwrite(out_rect, sbs)
    print(f"[save] {out_rect}")

# save
npz_path = os.path.join(OUT_DIR, "stereo_calib_stacked_v2.npz")
np.savez(
    npz_path,
    K1=K1s, dist1=dist1s, K2=K2s, dist2=dist2s,
    R=R, T=T, E=E, F=F,
    R1=R1, R2=R2, P1=P1, P2=P2, Q=Q,
    frameSize=np.array([frameSize[0], frameSize[1]], dtype=np.int32),
    chessboardSize=np.array([CHESSBOARD[0], CHESSBOARD[1]], dtype=np.int32),
    square_size_mm=float(SQUARE_SIZE_MM),
    RIGHT_MODE=RIGHT_MODE,
    rms_single_L=float(rmsL),
    rms_single_R=float(rmsR),
    rms_stereo=float(retStereo),
    n_pairs=int(accepted),
    model="opencv_pinhole_rational",
    note="stacked video halves upscaled; stereo uses USE_INTRINSIC_GUESS"
)
print(f"[save] {npz_path}")


[info] half frameSize after upscale: (3840, 1080)
[collect] accepted=20 at frame=19 | dbg=stereo_from_stacked_debug_v2\dbg_pairs\pair_020_f19.png
[collect] accepted=40 at frame=39 | dbg=stereo_from_stacked_debug_v2\dbg_pairs\pair_040_f39.png
[collect] accepted=60 at frame=59 | dbg=stereo_from_stacked_debug_v2\dbg_pairs\pair_060_f59.png
[collect] accepted=80 at frame=79 | dbg=stereo_from_stacked_debug_v2\dbg_pairs\pair_080_f79.png
[collect] accepted=100 at frame=99 | dbg=stereo_from_stacked_debug_v2\dbg_pairs\pair_100_f99.png
[collect] accepted=120 at frame=119 | dbg=stereo_from_stacked_debug_v2\dbg_pairs\pair_120_f119.png
[done] collected pairs: 120
[single] RMS L: 0.3387 px
[single] RMS R: 0.3771 px
[stereo] RMS: 6.3158 px
[stereo] baseline |T|: 390.547 mm
[stereo] T (mm): [-359.23255943  100.12724544  115.9900649 ]
[save] stereo_from_stacked_debug_v2\rectified_side_by_side.png
[save] stereo_from_stacked_debug_v2\stereo_calib_stacked_v2.npz


In [1]:
import os
import cv2 as cv
import numpy as np

# =========================
# USER SETTINGS
# =========================
LEFT_VIDEO  = "calib8_left.mp4"
RIGHT_VIDEO = "calib8_right.mp4"

OUT_DIR = "stereo_from_2videos_debug_v2"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "dbg_pairs"), exist_ok=True)

CHESSBOARD = (6, 5)        # (cols, rows)
SQUARE_SIZE_MM = 14.0

START_FRAME = 0
SKIP = 1
MAX_SAMPLES = 120
DEBUG_EVERY = 20

# If you flipped RIGHT video visually, set this
RIGHT_MODE = "id"          # "id", "flip_x", "flip_y", "flip_xy"

# Optional: upscale each frame to a fixed height (e.g., if export is 540p but you want 1080p)
UPSCALE_TO_H = None        # e.g. 1080, or None to disable

SUBPIX_WIN = (11, 11)
criteria_subpix = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 1e-3)
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 80, 1e-7)

INTRINSIC_FLAGS = cv.CALIB_RATIONAL_MODEL
STEREO_FLAGS = cv.CALIB_USE_INTRINSIC_GUESS

RECTIFY_ALPHA = 0.2


# =========================
# HELPERS
# =========================
def apply_mode(img, mode):
    if mode == "id": return img
    if mode == "flip_x": return cv.flip(img, 1)
    if mode == "flip_y": return cv.flip(img, 0)
    if mode == "flip_xy": return cv.flip(img, -1)
    raise ValueError(mode)

def build_objp(board_size, sq_mm):
    cols, rows = board_size
    objp = np.zeros((rows * cols, 3), np.float32)
    objp[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2)
    objp *= float(sq_mm)
    return objp

def reorder_corners_for_mode(corners, pattern, mode):
    cols, rows = pattern
    c = corners.reshape(rows, cols, 2).copy()
    if mode == "id":
        pass
    elif mode == "flip_x":
        c = c[:, ::-1, :]
    elif mode == "flip_y":
        c = c[::-1, :, :]
    elif mode == "flip_xy":
        c = c[::-1, ::-1, :]
    else:
        raise ValueError(mode)
    return c.reshape(-1, 1, 2).astype(np.float32)

def maybe_upscale(img, target_h):
    if target_h is None:
        return img
    h, w = img.shape[:2]
    if h == target_h:
        return img
    scale = target_h / float(h)
    new_w = int(round(w * scale))
    return cv.resize(img, (new_w, target_h), interpolation=cv.INTER_CUBIC)

def find_corners(gray, board):
    ok, corners = cv.findChessboardCornersSB(gray, board, flags=cv.CALIB_CB_NORMALIZE_IMAGE)
    if not ok or corners is None:
        ok, corners = cv.findChessboardCorners(
            gray, board,
            flags=cv.CALIB_CB_ADAPTIVE_THRESH | cv.CALIB_CB_NORMALIZE_IMAGE
        )
    if not ok or corners is None:
        return None
    corners = cv.cornerSubPix(gray, corners, SUBPIX_WIN, (-1, -1), criteria_subpix)
    return corners

def save_debug_pair(frameL, frameR, cornersL, cornersR, accepted, frame_idx):
    a = frameL.copy()
    b = frameR.copy()
    cv.drawChessboardCorners(a, CHESSBOARD, cornersL, True)
    cv.drawChessboardCorners(b, CHESSBOARD, cornersR, True)
    sbs = np.hstack([a, b])
    out = os.path.join(OUT_DIR, "dbg_pairs", f"pair_{accepted:03d}_f{frame_idx}.png")
    cv.imwrite(out, sbs)
    return out

def get_nframes(cap):
    n = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
    return n if n > 0 else None


# =========================
# RUN (2 separate videos)
# =========================
objp = build_objp(CHESSBOARD, SQUARE_SIZE_MM)
objpoints, imgpointsL, imgpointsR = [], [], []

capL = cv.VideoCapture(LEFT_VIDEO)
capR = cv.VideoCapture(RIGHT_VIDEO)
if not capL.isOpened():
    raise RuntimeError(f"Left video açılamadı: {LEFT_VIDEO}")
if not capR.isOpened():
    raise RuntimeError(f"Right video açılamadı: {RIGHT_VIDEO}")

# Optional sanity checks
nL = get_nframes(capL)
nR = get_nframes(capR)
if nL is not None and nR is not None:
    print(f"[info] frame counts: L={nL}, R={nR}")
    if nL != nR:
        raise RuntimeError("İki videonun frame sayısı farklı. (sync/export sorunu olabilir)")

# Seek start frame
capL.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)
capR.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)

retL, firstL = capL.read()
retR, firstR = capR.read()
if not retL:
    raise RuntimeError("Left ilk frame okunamadı.")
if not retR:
    raise RuntimeError("Right ilk frame okunamadı.")

firstR = apply_mode(firstR, RIGHT_MODE)

firstL = maybe_upscale(firstL, UPSCALE_TO_H)
firstR = maybe_upscale(firstR, UPSCALE_TO_H)

if firstL.shape[:2] != firstR.shape[:2]:
    raise RuntimeError("Up-scale sonrası L/R boyutu farklı çıktı. Videoların çözünürlükleri eşleşmiyor.")

frameSize = (firstL.shape[1], firstL.shape[0])  # (w,h)
print(f"[info] frameSize: {frameSize}")

# Reset again to START_FRAME for collection loop
capL.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)
capR.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)

accepted = 0
frame_idx = START_FRAME

while True:
    retL, frameL = capL.read()
    retR, frameR0 = capR.read()

    if (not retL) or (not retR):
        break

    if (frame_idx - START_FRAME) % SKIP != 0:
        frame_idx += 1
        continue

    frameR = apply_mode(frameR0, RIGHT_MODE)

    frameL = maybe_upscale(frameL, UPSCALE_TO_H)
    frameR = maybe_upscale(frameR, UPSCALE_TO_H)

    if frameL.shape[:2] != frameR.shape[:2]:
        raise RuntimeError(f"Frame size mismatch at frame {frame_idx}. L={frameL.shape[:2]}, R={frameR.shape[:2]}")

    grayL = cv.cvtColor(frameL, cv.COLOR_BGR2GRAY)
    grayR = cv.cvtColor(frameR, cv.COLOR_BGR2GRAY)

    cornersL = find_corners(grayL, CHESSBOARD)
    cornersR = find_corners(grayR, CHESSBOARD)

    if cornersL is not None and cornersR is not None:
        cornersR = reorder_corners_for_mode(cornersR, CHESSBOARD, RIGHT_MODE)

        objpoints.append(objp.copy())
        imgpointsL.append(cornersL)
        imgpointsR.append(cornersR)

        accepted += 1
        if accepted % DEBUG_EVERY == 0:
            out = save_debug_pair(frameL, frameR, cornersL, cornersR, accepted, frame_idx)
            print(f"[collect] accepted={accepted} at frame={frame_idx} | dbg={out}")

        if accepted >= MAX_SAMPLES:
            break

    frame_idx += 1

capL.release()
capR.release()

print(f"[done] collected pairs: {accepted}")
if accepted < 15:
    raise RuntimeError("Çok az pair. Görüntü netliği / chessboard boyutu / poz / ışık kontrol et.")

# --- Intrinsics ---
rmsL, K1, dist1, *_ = cv.calibrateCamera(
    objpoints, imgpointsL, frameSize, None, None,
    flags=INTRINSIC_FLAGS, criteria=criteria
)
rmsR, K2, dist2, *_ = cv.calibrateCamera(
    objpoints, imgpointsR, frameSize, None, None,
    flags=INTRINSIC_FLAGS, criteria=criteria
)

print(f"[single] RMS L: {rmsL:.4f} px")
print(f"[single] RMS R: {rmsR:.4f} px")

# --- Stereo ---
retStereo, K1s, dist1s, K2s, dist2s, R, T, E, F = cv.stereoCalibrate(
    objpoints, imgpointsL, imgpointsR,
    K1, dist1, K2, dist2, frameSize,
    criteria=criteria,
    flags=STEREO_FLAGS
)

print(f"[stereo] RMS: {retStereo:.4f} px")
print(f"[stereo] baseline |T|: {float(np.linalg.norm(T)):.3f} mm")
print(f"[stereo] T (mm): {T.ravel()}")

# --- Rectify ---
R1, R2, P1, P2, Q, *_ = cv.stereoRectify(
    K1s, dist1s, K2s, dist2s, frameSize, R, T, alpha=RECTIFY_ALPHA
)
map1L, map2L = cv.initUndistortRectifyMap(K1s, dist1s, R1, P1, frameSize, cv.CV_16SC2)
map1R, map2R = cv.initUndistortRectifyMap(K2s, dist2s, R2, P2, frameSize, cv.CV_16SC2)

# preview rectified (grab one frame)
capL = cv.VideoCapture(LEFT_VIDEO)
capR = cv.VideoCapture(RIGHT_VIDEO)
capL.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)
capR.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)
retL, fL = capL.read()
retR, fR0 = capR.read()
capL.release()
capR.release()

if retL and retR:
    fR = apply_mode(fR0, RIGHT_MODE)
    fL = maybe_upscale(fL, UPSCALE_TO_H)
    fR = maybe_upscale(fR, UPSCALE_TO_H)

    rectL = cv.remap(fL, map1L, map2L, cv.INTER_LINEAR)
    rectR = cv.remap(fR, map1R, map2R, cv.INTER_LINEAR)

    sbs = np.hstack([rectL, rectR])
    step = max(30, frameSize[1] // 18)
    for y in range(0, sbs.shape[0], step):
        cv.line(sbs, (0, y), (sbs.shape[1]-1, y), (0, 255, 0), 1)

    out_rect = os.path.join(OUT_DIR, "rectified_side_by_side.png")
    cv.imwrite(out_rect, sbs)
    print(f"[save] {out_rect}")

# save
npz_path = os.path.join(OUT_DIR, "stereo_calib_2videos_v2.npz")
np.savez(
    npz_path,
    K1=K1s, dist1=dist1s, K2=K2s, dist2=dist2s,
    R=R, T=T, E=E, F=F,
    R1=R1, R2=R2, P1=P1, P2=P2, Q=Q,
    frameSize=np.array([frameSize[0], frameSize[1]], dtype=np.int32),
    chessboardSize=np.array([CHESSBOARD[0], CHESSBOARD[1]], dtype=np.int32),
    square_size_mm=float(SQUARE_SIZE_MM),
    RIGHT_MODE=RIGHT_MODE,
    rms_single_L=float(rmsL),
    rms_single_R=float(rmsR),
    rms_stereo=float(retStereo),
    n_pairs=int(accepted),
    model="opencv_pinhole_rational",
    note="2 separate videos; stereo uses USE_INTRINSIC_GUESS"
)
print(f"[save] {npz_path}")


[info] frame counts: L=782, R=782
[info] frameSize: (1280, 720)
[collect] accepted=20 at frame=31 | dbg=stereo_from_2videos_debug_v2\dbg_pairs\pair_020_f31.png
[collect] accepted=40 at frame=51 | dbg=stereo_from_2videos_debug_v2\dbg_pairs\pair_040_f51.png
[collect] accepted=60 at frame=71 | dbg=stereo_from_2videos_debug_v2\dbg_pairs\pair_060_f71.png
[collect] accepted=80 at frame=91 | dbg=stereo_from_2videos_debug_v2\dbg_pairs\pair_080_f91.png
[collect] accepted=100 at frame=111 | dbg=stereo_from_2videos_debug_v2\dbg_pairs\pair_100_f111.png
[collect] accepted=120 at frame=131 | dbg=stereo_from_2videos_debug_v2\dbg_pairs\pair_120_f131.png
[done] collected pairs: 120
[single] RMS L: 0.1296 px
[single] RMS R: 0.1700 px
[stereo] RMS: 28.1749 px
[stereo] baseline |T|: 395.681 mm
[stereo] T (mm): [-207.57650777  114.64395706  316.75266791]
[save] stereo_from_2videos_debug_v2\rectified_side_by_side.png
[save] stereo_from_2videos_debug_v2\stereo_calib_2videos_v2.npz


In [8]:
import os
import cv2 as cv
import numpy as np

# =========================
# PATHS
# =========================
LEFT_VIDEO  = "test8_left.mp4"
RIGHT_VIDEO = "test8_right.mp4"
CALIB_NPZ   = "stereo_from_2videos_debug_v2/stereo_calib_2videos_v2.npz"

OUT_DIR = "track3d_threshold_out"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_NPZ = os.path.join(OUT_DIR, "tracked_xyz_mm.npz")
OUT_CSV = os.path.join(OUT_DIR, "tracked_xyz_mm.csv")
OUT_GIF = os.path.join(OUT_DIR, "tracked_boxes.gif")

# =========================
# RUN SETTINGS
# =========================
START_FRAME = 0
SKIP = 1
MAX_FRAMES = None          # None = to end
DRAW_SIDE_BY_SIDE = True
GIF_FPS = 15

# Kalibrasyonda UPSCALE_TO_H kullandıysan burada da aynı olmalı
UPSCALE_TO_H = None

# =========================
# PINK THRESHOLD SETTINGS (HSV)
# =========================
# OpenCV HSV: H in [0..179], S,V in [0..255]
# Pembe genelde 140-175 arası; bazen 160-179 daha iyi olur.
PINK_LO = (140, 60, 60)
PINK_HI = (179, 255, 255)

# Maske temizliği
MORPH_OPEN_IT = 1
MORPH_CLOSE_IT = 2
MIN_AREA = 30              # ROI içindeki pembe blob min alan (px)
USE_LARGEST_BLOB = True

# =========================
# HELPERS
# =========================
def apply_mode(img, mode):
    if mode == "id": return img
    if mode == "flip_x": return cv.flip(img, 1)
    if mode == "flip_y": return cv.flip(img, 0)
    if mode == "flip_xy": return cv.flip(img, -1)
    raise ValueError(mode)

def maybe_upscale(img, target_h):
    if target_h is None:
        return img
    h, w = img.shape[:2]
    if h == target_h:
        return img
    scale = target_h / float(h)
    new_w = int(round(w * scale))
    return cv.resize(img, (new_w, target_h), interpolation=cv.INTER_CUBIC)

def rectified_points_from_pixel(pts_px, K, dist, Rrect, Prect):
    pts = np.asarray(pts_px, dtype=np.float32).reshape(-1, 1, 2)
    und = cv.undistortPoints(pts, K, dist, R=Rrect, P=Prect)  # (N,1,2) rectified pixel coords
    return und.reshape(-1, 2)

def triangulate_rectified(P1, P2, pts1_rect_px, pts2_rect_px):
    p1 = np.asarray(pts1_rect_px, dtype=np.float64).T  # (2,N)
    p2 = np.asarray(pts2_rect_px, dtype=np.float64).T  # (2,N)
    X4 = cv.triangulatePoints(P1, P2, p1, p2)          # (4,N)
    X = (X4[:3] / X4[3]).T                             # (N,3)
    return X

def clamp_rect_to_image(rect, w, h):
    x, y, rw, rh = rect
    x = int(max(0, min(x, w-1)))
    y = int(max(0, min(y, h-1)))
    rw = int(max(1, min(rw, w-x)))
    rh = int(max(1, min(rh, h-y)))
    return (x, y, rw, rh)

def pink_centroid_in_roi(frame_bgr, roi, pink_lo, pink_hi):
    """
    Returns (u,v) in full-image pixel coords, plus debug mask stats.
    If not found, returns (np.nan, np.nan).
    """
    h, w = frame_bgr.shape[:2]
    x, y, rw, rh = clamp_rect_to_image(roi, w, h)
    patch = frame_bgr[y:y+rh, x:x+rw]

    hsv = cv.cvtColor(patch, cv.COLOR_BGR2HSV)
    mask = cv.inRange(hsv, np.array(pink_lo, np.uint8), np.array(pink_hi, np.uint8))

    # Morph cleanup
    k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (3, 3))
    for _ in range(MORPH_OPEN_IT):
        mask = cv.morphologyEx(mask, cv.MORPH_OPEN, k)
    for _ in range(MORPH_CLOSE_IT):
        mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, k)

    # Find blobs
    cnts, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return (np.nan, np.nan), mask, (x, y, rw, rh), 0

    # Choose largest blob (usually correct)
    if USE_LARGEST_BLOB:
        cnt = max(cnts, key=cv.contourArea)
        area = float(cv.contourArea(cnt))
        if area < MIN_AREA:
            return (np.nan, np.nan), mask, (x, y, rw, rh), area

        M = cv.moments(cnt)
        if M["m00"] == 0:
            return (np.nan, np.nan), mask, (x, y, rw, rh), area

        cx = (M["m10"] / M["m00"]) + x
        cy = (M["m01"] / M["m00"]) + y
        return (float(cx), float(cy)), mask, (x, y, rw, rh), area

    # (İstersen başka seçim stratejileri de ekleriz)
    return (np.nan, np.nan), mask, (x, y, rw, rh), 0

# =========================
# LOAD CALIB
# =========================
cal = np.load(CALIB_NPZ, allow_pickle=True)

K1 = cal["K1"]
dist1 = cal["dist1"].reshape(-1, 1)
K2 = cal["K2"]
dist2 = cal["dist2"].reshape(-1, 1)
R1 = cal["R1"]
R2 = cal["R2"]
P1 = cal["P1"]
P2 = cal["P2"]

RIGHT_MODE = str(cal.get("RIGHT_MODE", "id"))
frameSize = tuple(cal["frameSize"].astype(int))  # (w,h)

print("[calib] Loaded.")
print("[calib] RIGHT_MODE:", RIGHT_MODE)
print("[calib] frameSize (w,h):", frameSize)

# =========================
# OPEN FIRST FRAME
# =========================
capL = cv.VideoCapture(LEFT_VIDEO)
capR = cv.VideoCapture(RIGHT_VIDEO)
if not capL.isOpened(): raise RuntimeError(f"Left video açılamadı: {LEFT_VIDEO}")
if not capR.isOpened(): raise RuntimeError(f"Right video açılamadı: {RIGHT_VIDEO}")

capL.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)
capR.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)

retL, firstL = capL.read()
retR, firstR0 = capR.read()
if not retL or not retR:
    raise RuntimeError("İlk frameler okunamadı.")

firstR = apply_mode(firstR0, RIGHT_MODE)
firstL = maybe_upscale(firstL, UPSCALE_TO_H)
firstR = maybe_upscale(firstR, UPSCALE_TO_H)

if firstL.shape[:2] != firstR.shape[:2]:
    raise RuntimeError(f"L/R size mismatch after mode/upscale: L={firstL.shape[:2]} R={firstR.shape[:2]}")

print("[info] First frame size:", (firstL.shape[1], firstL.shape[0]))

# =========================
# ROI SELECTION
# =========================
n = int(input("Kaç tane pembe noktayı track edeceğiz? (N): ").strip())
if n <= 0:
    raise RuntimeError("N pozitif olmalı.")

print("\n[ROI] Sol frame açılacak. Her marker için ROI çizip ENTER'a bas.\n")
roisL = []
for i in range(n):
    r = cv.selectROI(f"LEFT - ROI for point #{i+1}", firstL, fromCenter=False, showCrosshair=True)
    cv.destroyWindow(f"LEFT - ROI for point #{i+1}")
    if r is None or r[2] == 0 or r[3] == 0:
        raise RuntimeError("ROI seçimi iptal edildi/boş.")
    roisL.append(tuple(map(int, r)))

print("\n[ROI] Sağ frame açılacak. Aynı sırayla (sol ile eşleşecek) ROI seç.\n")
roisR = []
for i in range(n):
    r = cv.selectROI(f"RIGHT - ROI for point #{i+1}", firstR, fromCenter=False, showCrosshair=True)
    cv.destroyWindow(f"RIGHT - ROI for point #{i+1}")
    if r is None or r[2] == 0 or r[3] == 0:
        raise RuntimeError("ROI seçimi iptal edildi/boş.")
    roisR.append(tuple(map(int, r)))

# Reset captures to start
capL.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)
capR.set(cv.CAP_PROP_POS_FRAMES, START_FRAME)

# =========================
# MAIN LOOP
# =========================
frames = []
UVL_all = []
UVR_all = []
OK_all  = []
XYZ_all = []
gif_frames_rgb = []

frame_idx = START_FRAME
count = 0

print("\n[run] Threshold tracking başladı...\n")
while True:
    if MAX_FRAMES is not None and count >= MAX_FRAMES:
        break

    retL, frameL = capL.read()
    retR, frameR0 = capR.read()
    if not retL or not retR:
        break

    if (frame_idx - START_FRAME) % SKIP != 0:
        frame_idx += 1
        continue

    frameR = apply_mode(frameR0, RIGHT_MODE)
    frameL = maybe_upscale(frameL, UPSCALE_TO_H)
    frameR = maybe_upscale(frameR, UPSCALE_TO_H)

    ptsL = np.full((n, 2), np.nan, dtype=np.float32)
    ptsR = np.full((n, 2), np.nan, dtype=np.float32)
    ok  = np.zeros((n,), dtype=np.uint8)

    drawL = frameL.copy()
    drawR = frameR.copy()

    # find centroids in each ROI
    for i in range(n):
        (uL, vL), maskL, roiL_clamped, areaL = pink_centroid_in_roi(drawL, roisL[i], PINK_LO, PINK_HI)
        (uR, vR), maskR, roiR_clamped, areaR = pink_centroid_in_roi(drawR, roisR[i], PINK_LO, PINK_HI)

        ptsL[i] = (uL, vL)
        ptsR[i] = (uR, vR)

        if np.isfinite(uL) and np.isfinite(uR):
            ok[i] = 1

        # draw ROI
        x,y,w,h = roiL_clamped
        cv.rectangle(drawL, (x,y), (x+w, y+h), (255, 255, 0), 2)
        x,y,w,h = roiR_clamped
        cv.rectangle(drawR, (x,y), (x+w, y+h), (255, 255, 0), 2)

        # draw centroid if ok
        if ok[i]:
            cv.circle(drawL, (int(uL), int(vL)), 5, (0,255,0), -1)
            cv.putText(drawL, f"#{i+1}", (int(uL)+6, int(vL)-6), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
            cv.circle(drawR, (int(uR), int(vR)), 5, (0,255,0), -1)
            cv.putText(drawR, f"#{i+1}", (int(uR)+6, int(vR)-6), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
        else:
            # failed marker label
            cv.putText(drawL, f"#{i+1} LOST", (roisL[i][0], max(15, roisL[i][1]-5)),
                       cv.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,255), 2)
            cv.putText(drawR, f"#{i+1} LOST", (roisR[i][0], max(15, roisR[i][1]-5)),
                       cv.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,255), 2)

    # triangulate
    XYZ = np.full((n, 3), np.nan, dtype=np.float64)
    valid = ok.astype(bool) & np.isfinite(ptsL).all(axis=1) & np.isfinite(ptsR).all(axis=1)

    if np.any(valid):
        ptsL_rect = rectified_points_from_pixel(ptsL[valid], K1, dist1, R1, P1)
        ptsR_rect = rectified_points_from_pixel(ptsR[valid], K2, dist2, R2, P2)
        X_valid = triangulate_rectified(P1, P2, ptsL_rect, ptsR_rect)  # mm
        XYZ[valid] = X_valid

        # write XYZ on frame
        idxs = np.where(valid)[0]
        for j, mi in enumerate(idxs):
            X, Y, Z = XYZ[mi]
            uL, vL = ptsL[mi]
            cv.putText(drawL, f"X{X:.1f} Y{Y:.1f} Z{Z:.1f} mm",
                       (int(uL)+6, int(vL)+18), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)

    frames.append(frame_idx)
    UVL_all.append(ptsL.copy())
    UVR_all.append(ptsR.copy())
    OK_all.append(ok.copy())
    XYZ_all.append(XYZ.copy())

    # gif frame
    if DRAW_SIDE_BY_SIDE:
        sbs = np.hstack([drawL, drawR])
        cv.putText(sbs, f"frame {frame_idx}", (15, 30), cv.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        rgb = cv.cvtColor(sbs, cv.COLOR_BGR2RGB)
    else:
        cv.putText(drawL, f"frame {frame_idx}", (15, 30), cv.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        rgb = cv.cvtColor(drawL, cv.COLOR_BGR2RGB)

    gif_frames_rgb.append(rgb)

    count += 1
    if count % 50 == 0:
        print(f"[run] processed {count} frames (frame_idx={frame_idx})")

    frame_idx += 1

capL.release()
capR.release()
print(f"\n[done] Total processed frames: {count}")

# =========================
# SAVE (NPZ + CSV)
# =========================
frames = np.array(frames, dtype=np.int32)
UVL_all = np.stack(UVL_all, axis=0)  # (T,N,2)
UVR_all = np.stack(UVR_all, axis=0)
OK_all  = np.stack(OK_all, axis=0)   # (T,N)
XYZ_all = np.stack(XYZ_all, axis=0)  # (T,N,3)

np.savez(
    OUT_NPZ,
    frames=frames,
    UVL_px=UVL_all,
    UVR_px=UVR_all,
    OK=OK_all,
    XYZ_mm=XYZ_all,
    left_video=LEFT_VIDEO,
    right_video=RIGHT_VIDEO,
    calib_npz=CALIB_NPZ,
    right_mode=RIGHT_MODE,
    upscale_to_h=UPSCALE_TO_H,
    pink_lo=np.array(PINK_LO),
    pink_hi=np.array(PINK_HI),
)

with open(OUT_CSV, "w", encoding="utf-8") as f:
    f.write("frame,marker,ok,uL,vL,uR,vR,X_mm,Y_mm,Z_mm\n")
    T = XYZ_all.shape[0]
    for ti in range(T):
        fr = int(frames[ti])
        for mi in range(n):
            okv = int(OK_all[ti, mi])
            uL, vL = UVL_all[ti, mi]
            uR, vR = UVR_all[ti, mi]
            X, Y, Z = XYZ_all[ti, mi]
            f.write(f"{fr},{mi+1},{okv},{uL:.3f},{vL:.3f},{uR:.3f},{vR:.3f},{X:.6f},{Y:.6f},{Z:.6f}\n")

print("[save]", OUT_NPZ)
print("[save]", OUT_CSV)



[calib] Loaded.
[calib] RIGHT_MODE: id
[calib] frameSize (w,h): (1280, 720)
[info] First frame size: (1280, 720)
Kaç tane pembe noktayı track edeceğiz? (N): 3

[ROI] Sol frame açılacak. Her marker için ROI çizip ENTER'a bas.


[ROI] Sağ frame açılacak. Aynı sırayla (sol ile eşleşecek) ROI seç.


[run] Threshold tracking başladı...

[run] processed 50 frames (frame_idx=49)
[run] processed 100 frames (frame_idx=99)
[run] processed 150 frames (frame_idx=149)
[run] processed 200 frames (frame_idx=199)
[run] processed 250 frames (frame_idx=249)
[run] processed 300 frames (frame_idx=299)
[run] processed 350 frames (frame_idx=349)
[run] processed 400 frames (frame_idx=399)
[run] processed 450 frames (frame_idx=449)
[run] processed 500 frames (frame_idx=499)

[done] Total processed frames: 542
[save] track3d_threshold_out\tracked_xyz_mm.npz
[save] track3d_threshold_out\tracked_xyz_mm.csv


C:\Users\edanu\AppData\Roaming\Python\Python39\site-packages\numpy\_typing\_scalars.py:12: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  _BoolLike_co = Union[bool, np.bool]


RuntimeError: imageio yok. `pip install imageio` kurup tekrar dene.

In [9]:
OUT_MP4 = os.path.join(OUT_DIR, "tracked_boxes.mp4")

h, w, _ = gif_frames_rgb[0].shape
fourcc = cv.VideoWriter_fourcc(*"mp4v")
vw = cv.VideoWriter(OUT_MP4, fourcc, GIF_FPS, (w, h))

for rgb in gif_frames_rgb:
    bgr = cv.cvtColor(rgb, cv.COLOR_RGB2BGR)
    vw.write(bgr)

vw.release()
print("[save]", OUT_MP4)


[save] track3d_threshold_out\tracked_boxes.mp4


In [5]:
!pip install opencv-contrib-python

Defaulting to user installation because normal site-packages is not writeable
  Using cached opencv_contrib_python-4.13.0.90-cp37-abi3-win_amd64.whl (46.5 MB)
